# Unified Validation Pipeline

This notebook rebuilds the core modeling workflow in one place using the raw CSV as the only source-of-truth input.

## Scope
- Load the dataset from `data/smart_grid_stability_augmented.csv`
- Apply feature engineering
- Apply consensus feature selection on the training split only
- Evaluate baseline models with scaler selection
- Run Grey Wolf Optimization (GWO)
- Run Tree-structured Parzen Estimator (TPE)
- Produce train-side out-of-fold metrics, held-out test metrics, and confidence intervals for the 8 evaluation metrics


 ## imports and constants

In [1]:
import json
import math
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import RFE, SelectFromModel, VarianceThreshold, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, QuantileTransformer, RobustScaler, StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "smart_grid_stability_augmented.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from the current notebook working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "smart_grid_stability_augmented.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables" / "sum_logic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
N_TRIALS_PER_MODEL = 50
GWO_POPULATION_SIZE = 10
GWO_ITERATIONS = 4  # 10 init + 4*10 iterations = 50 total, identical to TPE budget
N_BOOTSTRAP = 2000
VOTE_THRESHOLD = 3
VAR_THRESHOLD = 0.01
CORR_THRESHOLD = 0.95
TOP_K_FRACTION = 0.50

TARGET_COLUMN = "stabf"
LEAKAGE_COLUMN = "stab"
METRICS = ["Accuracy", "Precision", "Recall", "F1", "AUC", "LogLoss", "Kappa", "MCC"]
METHOD_ORDER = ["Baseline", "GWO", "TPE"]
MODEL_ORDER = ["LR", "LDA", "QDA", "NB", "KNN", "LinearSVC", "SVM", "AdaBoost", "RF", "GB", "XGB", "LGBM", "CB", "SGD"]

SCALER_FACTORIES = {
    "Raw": lambda: None,
    "Standard": StandardScaler,
    "MinMax": MinMaxScaler,
    "Robust": RobustScaler,
    "Power": PowerTransformer,
    "Quantile": lambda: QuantileTransformer(output_distribution="normal", random_state=RANDOM_STATE),
}

MODEL_LABELS = {
    "LR": "LogisticRegression",
    "LDA": "LDA",
    "QDA": "QDA",
    "NB": "NaiveBayes",
    "KNN": "KNN",
    "LinearSVC": "LinearSVC",
    "SVM": "SVC",
    "AdaBoost": "AdaBoost",
    "RF": "RandomForest",
    "GB": "GradientBoosting",
    "XGB": "XGBoost",
    "LGBM": "LightGBM",
    "CB": "CatBoost",
    "SGD": "SGD",
}

SEARCH_SPACES = {
    "LR": {
        "C": {"type": "float", "range": [1e-4, 100.0], "log": True},
        "penalty": {"type": "categorical", "choices": ["l1", "l2"]},
    },
    "SGD": {
        "loss": {"type": "categorical", "choices": ["log_loss", "modified_huber"]},
        "penalty": {"type": "categorical", "choices": ["l1", "l2", "elasticnet"]},
        "alpha": {"type": "float", "range": [1e-6, 1e-1], "log": True},
        "l1_ratio": {"type": "float", "range": [0.0, 1.0], "log": False},
    },
    "LDA": {
        "solver": {"type": "categorical", "choices": ["svd", "lsqr", "eigen"]},
    },
    "QDA": {
        "reg_param": {"type": "float", "range": [0.0, 1.0], "log": False},
    },
    "NB": {
        "var_smoothing": {"type": "float", "range": [1e-12, 1e-6], "log": True},
    },
    "KNN": {
        "n_neighbors": {"type": "int", "range": [1, 50], "log": False},
        "weights": {"type": "categorical", "choices": ["uniform", "distance"]},
        "metric": {"type": "categorical", "choices": ["euclidean", "manhattan", "minkowski"]},
        "p": {"type": "int", "range": [1, 5], "log": False},
    },
    "LinearSVC": {
        "C": {"type": "float", "range": [1e-4, 100.0], "log": True},
        "loss": {"type": "categorical", "choices": ["hinge", "squared_hinge"]},
    },
    "SVM": {
        "kernel": {"type": "categorical", "choices": ["rbf", "poly", "sigmoid"]},
        "C": {"type": "float", "range": [1e-3, 1000.0], "log": True},
        "gamma": {"type": "categorical", "choices": ["scale", "auto"]},
        "degree": {"type": "int", "range": [2, 5], "log": False},
        "coef0": {"type": "float", "range": [0.0, 1.0], "log": False},
    },
    "RF": {
        "n_estimators": {"type": "int", "range": [50, 500], "log": False},
        "max_depth": {"type": "int", "range": [3, 30], "log": False},
        "min_samples_split": {"type": "int", "range": [2, 20], "log": False},
        "min_samples_leaf": {"type": "int", "range": [1, 10], "log": False},
        "max_features": {"type": "categorical", "choices": ["sqrt", "log2", None]},
    },
    "AdaBoost": {
        "n_estimators": {"type": "int", "range": [50, 500], "log": False},
        "learning_rate": {"type": "float", "range": [0.01, 2.0], "log": True},
        "algorithm": {"type": "categorical", "choices": ["SAMME", "SAMME.R"]},
    },
    "GB": {
        "n_estimators": {"type": "int", "range": [50, 500], "log": False},
        "learning_rate": {"type": "float", "range": [0.01, 0.5], "log": True},
        "max_depth": {"type": "int", "range": [3, 15], "log": False},
        "min_samples_split": {"type": "int", "range": [2, 20], "log": False},
        "min_samples_leaf": {"type": "int", "range": [1, 10], "log": False},
        "subsample": {"type": "float", "range": [0.5, 1.0], "log": False},
    },
    "XGB": {
        "n_estimators": {"type": "int", "range": [50, 500], "log": False},
        "learning_rate": {"type": "float", "range": [0.01, 0.5], "log": True},
        "max_depth": {"type": "int", "range": [3, 15], "log": False},
        "min_child_weight": {"type": "int", "range": [1, 10], "log": False},
        "subsample": {"type": "float", "range": [0.5, 1.0], "log": False},
        "colsample_bytree": {"type": "float", "range": [0.5, 1.0], "log": False},
        "gamma": {"type": "float", "range": [0.0, 5.0], "log": False},
        "reg_alpha": {"type": "float", "range": [1e-8, 10.0], "log": True},
        "reg_lambda": {"type": "float", "range": [1e-8, 10.0], "log": True},
    },
    "LGBM": {
        "n_estimators": {"type": "int", "range": [50, 500], "log": False},
        "learning_rate": {"type": "float", "range": [0.01, 0.5], "log": True},
        "max_depth": {"type": "int", "range": [3, 15], "log": False},
        "num_leaves": {"type": "int", "range": [20, 150], "log": False},
        "min_child_samples": {"type": "int", "range": [5, 100], "log": False},
        "subsample": {"type": "float", "range": [0.5, 1.0], "log": False},
        "colsample_bytree": {"type": "float", "range": [0.5, 1.0], "log": False},
        "reg_alpha": {"type": "float", "range": [1e-8, 10.0], "log": True},
        "reg_lambda": {"type": "float", "range": [1e-8, 10.0], "log": True},
    },
    "CB": {
        "iterations": {"type": "int", "range": [50, 500], "log": False},
        "learning_rate": {"type": "float", "range": [0.01, 0.5], "log": True},
        "depth": {"type": "int", "range": [3, 10], "log": False},
        "l2_leaf_reg": {"type": "float", "range": [1e-8, 10.0], "log": True},
        "border_count": {"type": "int", "range": [32, 255], "log": False},
        "bagging_temperature": {"type": "float", "range": [0.0, 10.0], "log": False},
    },
}

CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {DATA_PATH}")
print(f"Results path: {RESULTS_DIR}")
print(f"Optimizer search spaces defined for {len(SEARCH_SPACES)} models.")

c:\Users\omar8\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\omar8\Desktop\2026\Master & Research\smart-grid-stability-v01
Raw data path: C:\Users\omar8\Desktop\2026\Master & Research\smart-grid-stability-v01\data\smart_grid_stability_augmented.csv
Results path: C:\Users\omar8\Desktop\2026\Master & Research\smart-grid-stability-v01\results\tables\sum_logic
Optimizer search spaces defined for 14 models.


## system resource setup

In [3]:
# =====================================================================
# SYSTEM RESOURCE DETECTION & OPTIMIZATION
# =====================================================================

import psutil
import gc
import os

# Detect system resources
CPU_COUNT = os.cpu_count() or 4
AVAILABLE_MEMORY_GB = psutil.virtual_memory().available / (1024**3)
TOTAL_MEMORY_GB = psutil.virtual_memory().total / (1024**3)
MEMORY_PERCENT_AVAILABLE = (AVAILABLE_MEMORY_GB / TOTAL_MEMORY_GB) * 100

# Adaptive n_jobs based on memory and CPU
def compute_optimal_n_jobs(cpu_count: int, available_memory_gb: float, operation: str = "cv") -> int:
    """
    Compute optimal n_jobs to balance parallelization with memory constraints.
    
    Rules:
    - CV/Training: Use min(cpu_count - 2, cpu_available * 0.7)
    - Optimization: Use min(cpu_count - 2, 4) to prevent worker exhaustion
    - If <4GB available: Use 1 (single process only)
    """
    reserved_cores = 2
    available_cores = max(1, cpu_count - reserved_cores)
    
    if available_memory_gb < 4:
        return 1  # Single process if memory critical
    elif available_memory_gb < 8:
        return min(2, available_cores)
    elif operation == "optimization":
        return min(3, available_cores)  # Conservative for optimization loops
    else:
        return min(4, available_cores)  # Regular CV/training

N_JOBS_CV = compute_optimal_n_jobs(CPU_COUNT, AVAILABLE_MEMORY_GB, "cv")
N_JOBS_OPT = compute_optimal_n_jobs(CPU_COUNT, AVAILABLE_MEMORY_GB, "optimization")

print("="*80)
print("SYSTEM RESOURCE ANALYSIS & OPTIMIZATION")
print("="*80)
print(f"CPU Cores Available:        {CPU_COUNT}")
print(f"Total Memory:               {TOTAL_MEMORY_GB:.2f} GB")
print(f"Available Memory:           {AVAILABLE_MEMORY_GB:.2f} GB ({MEMORY_PERCENT_AVAILABLE:.1f}%)")
print(f"Optimized n_jobs (CV):      {N_JOBS_CV}")
print(f"Optimized n_jobs (OPT):     {N_JOBS_OPT}")
print("="*80)

# Resource monitoring class
class ResourceMonitor:
    def __init__(self, label: str = "Operation"):
        self.label = label
        self.start_time = None
        self.start_memory = None
        self.peak_memory = None
    
    def __enter__(self):
        self.start_time = time.perf_counter()
        self.start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024**2)  # MB
        return self
    
    def __exit__(self, *args):
        elapsed = time.perf_counter() - self.start_time
        current_memory = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        memory_delta = current_memory - self.start_memory
        print(f"  [{self.label}] Time: {elapsed:7.2f}s | Memory Δ: {memory_delta:+7.1f}MB | Peak: {current_memory:7.1f}MB")
        gc.collect()  # Explicit garbage collection

# Memory profiling decorator
def profile_memory(func):
    def wrapper(*args, **kwargs):
        gc.collect()  # Clean before
        start_mem = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        result = func(*args, **kwargs)
        end_mem = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        print(f"  └─ Memory change: {end_mem - start_mem:+.1f}MB")
        return result
    return wrapper

print("\n✓ Resource monitoring ready. Using adaptive parallelization.")
print("✓ Garbage collection enabled between major operations.")

SYSTEM RESOURCE ANALYSIS & OPTIMIZATION
CPU Cores Available:        8
Total Memory:               15.73 GB
Available Memory:           4.56 GB (29.0%)
Optimized n_jobs (CV):      2
Optimized n_jobs (OPT):     2

✓ Resource monitoring ready. Using adaptive parallelization.
✓ Garbage collection enabled between major operations.


## data load, engineering, selection

In [3]:
def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    tau_cols = ["tau1", "tau2", "tau3", "tau4"]
    g_cols = ["g1", "g2", "g3", "g4"]

    X["tau_mean"] = X[tau_cols].mean(axis=1)
    X["g_sum"] = X[g_cols].sum(axis=1)
    X["tau1_g1"] = X["tau1"] * X["g1"]
    X["tau2_g2"] = X["tau2"] * X["g2"]
    X["tau3_g3"] = X["tau3"] * X["g3"]
    X["tau4_g4"] = X["tau4"] * X["g4"]
    X["g_tau_ratio"] = X["g_sum"] / (X["tau_mean"] + 1e-5)
    return X


def consensus_feature_selection(X_train_input: pd.DataFrame, y_train_input: pd.Series) -> Tuple[List[str], pd.DataFrame]:
    votes = pd.DataFrame(index=X_train_input.columns)
    feature_select_jobs = max(1, min(N_JOBS_CV, 2))

    var_selector = VarianceThreshold(threshold=VAR_THRESHOLD)
    var_selector.fit(X_train_input)
    var_features = X_train_input.columns[var_selector.get_support()].tolist()
    votes["Variance"] = votes.index.isin(var_features).astype(int)

    corr_matrix = X_train_input[var_features].corr().abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper_triangle.columns if (upper_triangle[column] > CORR_THRESHOLD).any()]
    corr_features = [feature for feature in var_features if feature not in to_drop]
    votes["Correlation"] = votes.index.isin(corr_features).astype(int)

    X_corr = X_train_input[corr_features]
    X_corr_scaled = StandardScaler().fit_transform(X_corr)
    k_best = max(1, int(len(corr_features) * TOP_K_FRACTION))

    mi_scores = mutual_info_classif(X_corr_scaled, y_train_input, random_state=RANDOM_STATE)
    mi_features = X_corr.columns[np.argsort(mi_scores)[-k_best:]].tolist()
    votes["Mutual_Info"] = votes.index.isin(mi_features).astype(int)

    f_scores = f_classif(X_corr_scaled, y_train_input)[0]
    anova_features = X_corr.columns[np.argsort(f_scores)[-k_best:]].tolist()
    votes["ANOVA"] = votes.index.isin(anova_features).astype(int)

    rfe_selector = RFE(
        estimator=RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=feature_select_jobs),
        n_features_to_select=k_best,
    )
    rfe_selector.fit(X_corr_scaled, y_train_input)
    rfe_features = X_corr.columns[rfe_selector.support_].tolist()
    votes["RFE"] = votes.index.isin(rfe_features).astype(int)

    embedded_selector = SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=feature_select_jobs)
    )
    embedded_selector.fit(X_corr_scaled, y_train_input)
    embedded_features = X_corr.columns[embedded_selector.get_support()].tolist()
    votes["Embedded"] = votes.index.isin(embedded_features).astype(int)

    votes["Total"] = votes.sum(axis=1)
    selected_features = votes[votes["Total"] >= VOTE_THRESHOLD].index.tolist()
    return selected_features, votes.sort_values(["Total"], ascending=False)


def summarize_integrity(df: pd.DataFrame, X_engineered: pd.DataFrame, y_binary: pd.Series, selected: List[str]) -> pd.DataFrame:
    reproducible_split = train_test_split(
        X_engineered,
        y_binary,
        test_size=TEST_SIZE,
        stratify=y_binary,
        random_state=RANDOM_STATE,
    )
    reproducible = reproducible_split[0].index.equals(X_train_full.index) and reproducible_split[1].index.equals(X_test_full.index)

    summary = pd.DataFrame(
        [
            {"Check": "Raw rows", "Value": len(df)},
            {"Check": "Raw columns", "Value": df.shape[1]},
            {"Check": "Raw duplicate rows", "Value": int(df.duplicated().sum())},
            {"Check": "Raw missing values", "Value": int(df.isna().sum().sum())},
            {"Check": "Engineered feature count", "Value": X_engineered.shape[1]},
            {"Check": "Selected feature count", "Value": len(selected)},
            {"Check": "Train rows", "Value": len(X_train_full)},
            {"Check": "Test rows", "Value": len(X_test_full)},
            {"Check": "Train positive rate", "Value": float(y_train.mean())},
            {"Check": "Test positive rate", "Value": float(y_test.mean())},
            {"Check": "Split reproducible", "Value": bool(reproducible)},
        ]
    )
    return summary


df = pd.read_csv(DATA_PATH)
y = (df[TARGET_COLUMN] == "unstable").astype(int)
X_raw = df.drop(columns=[TARGET_COLUMN, LEAKAGE_COLUMN])
X_engineered = engineer_features(X_raw)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_engineered,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

selected_features, feature_votes = consensus_feature_selection(X_train_full, y_train)
X_train = X_train_full[selected_features].copy()
X_test = X_test_full[selected_features].copy()

reference_selected_path = PROJECT_ROOT / "results" / "tables" / "selected_features.json"
reference_selected_features = None
if reference_selected_path.exists():
    reference_selected_features = json.loads(reference_selected_path.read_text())

integrity_summary = summarize_integrity(df, X_engineered, y, selected_features)
validation_payload = {
    "selected_features": selected_features,
    "reference_selected_features": reference_selected_features,
    "matches_reference_selection": selected_features == reference_selected_features if reference_selected_features is not None else None,
}

feature_votes.to_csv(RESULTS_DIR / "feature_votes.csv")
pd.DataFrame({"feature": selected_features}).to_csv(RESULTS_DIR / "selected_features.csv", index=False)
integrity_summary.to_csv(RESULTS_DIR / "data_integrity_summary.csv", index=False)
with open(RESULTS_DIR / "validation_payload.json", "w", encoding="utf-8") as handle:
    json.dump(validation_payload, handle, indent=2)

assert df.isna().sum().sum() == 0, "Raw dataset contains missing values."
assert not X_train.isna().any().any(), "Training design matrix contains missing values."
assert not X_test.isna().any().any(), "Test design matrix contains missing values."
assert len(selected_features) > 0, "Feature selection returned no features."

print(f"Loaded raw dataset with shape {df.shape}")
print(f"Engineered feature matrix shape: {X_engineered.shape}")
print(f"Selected feature matrix shape: train={X_train.shape}, test={X_test.shape}")
print(f"Selection matches reference artifact: {validation_payload['matches_reference_selection']}")
display(integrity_summary)
display(feature_votes.head(10))

Loaded raw dataset with shape (60000, 14)
Engineered feature matrix shape: (60000, 19)
Selected feature matrix shape: train=(48000, 13), test=(12000, 13)
Selection matches reference artifact: True


,Check,Value
0,Raw rows,60000
1,Raw columns,14
2,Raw duplicate rows,0
3,Raw missing values,0
4,Engineered feature count,19
5,Selected feature count,13
6,Train rows,48000
7,Test rows,12000
8,Train positive rate,0.638
9,Test positive rate,0.638


,Variance,Correlation,Mutual_Info,ANOVA,RFE,Embedded,Total
g_sum,1,1,1,1,1,1,6
tau1_g1,1,1,1,1,1,1,6
tau2_g2,1,1,1,1,1,1,6
tau_mean,1,1,1,1,1,1,6
tau3_g3,1,1,1,1,1,1,6
tau1,1,1,1,0,1,1,5
tau3,1,1,0,1,1,1,5
tau4_g4,1,1,0,1,1,1,5
tau4,1,1,0,1,1,1,5
tau2,1,1,0,1,0,0,3


## model and metrics utilities

In [4]:
def get_scaler(scaler_name: str):
    factory = SCALER_FACTORIES[scaler_name]
    return None if factory is None or scaler_name == "Raw" else factory()


def build_model(model_code: str, params: Optional[Dict[str, Any]] = None):
    params = dict(params or {})

    if model_code == "LR":
        penalty = params.pop("penalty", "l2")
        solver = params.pop("solver", "saga")
        if penalty == "l1" and solver not in {"liblinear", "saga"}:
            solver = "saga"
        return LogisticRegression(
            C=params.pop("C", 1.0),
            penalty=penalty,
            solver=solver,
            max_iter=params.pop("max_iter", 2000),
            random_state=params.pop("random_state", RANDOM_STATE),
            **params,
        )

    if model_code == "LDA":
        solver = params.pop("solver", "svd")
        shrinkage = params.pop("shrinkage", None)
        if solver == "svd":
            return LinearDiscriminantAnalysis(solver=solver, **params)
        return LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage, **params)

    if model_code == "QDA":
        return QuadraticDiscriminantAnalysis(reg_param=params.pop("reg_param", 0.0), **params)

    if model_code == "NB":
        return GaussianNB(var_smoothing=params.pop("var_smoothing", 1e-9), **params)

    if model_code == "KNN":
        return KNeighborsClassifier(
            n_neighbors=params.pop("n_neighbors", 5),
            weights=params.pop("weights", "uniform"),
            metric=params.pop("metric", "minkowski"),
            p=params.pop("p", 2),
            **params,
        )

    if model_code == "LinearSVC":
        return LinearSVC(
            C=params.pop("C", 1.0),
            loss=params.pop("loss", "squared_hinge"),
            dual=params.pop("dual", "auto"),
            max_iter=params.pop("max_iter", 5000),
            random_state=params.pop("random_state", RANDOM_STATE),
            **params,
        )

    if model_code == "SVM":
        return SVC(
            kernel=params.pop("kernel", "rbf"),
            C=params.pop("C", 1.0),
            gamma=params.pop("gamma", "scale"),
            degree=params.pop("degree", 3),
            coef0=params.pop("coef0", 0.0),
            probability=True,
            random_state=RANDOM_STATE,
            **params,
        )

    if model_code == "AdaBoost":
        algorithm = params.pop("algorithm", "SAMME")
        if algorithm == "SAMME.R":
            algorithm = "SAMME"
        return AdaBoostClassifier(
            n_estimators=params.pop("n_estimators", 50),
            learning_rate=params.pop("learning_rate", 1.0),
            algorithm=algorithm,
            random_state=params.pop("random_state", RANDOM_STATE),
            **params,
        )

    if model_code == "RF":
        max_features = params.pop("max_features", "sqrt")
        if max_features == "None":
            max_features = None
        return RandomForestClassifier(
            n_estimators=params.pop("n_estimators", 100),
            max_depth=params.pop("max_depth", None),
            min_samples_split=params.pop("min_samples_split", 2),
            min_samples_leaf=params.pop("min_samples_leaf", 1),
            max_features=max_features,
            random_state=params.pop("random_state", RANDOM_STATE),
            n_jobs=N_JOBS_OPT,
            **params,
        )

    if model_code == "GB":
        return GradientBoostingClassifier(
            n_estimators=params.pop("n_estimators", 100),
            learning_rate=params.pop("learning_rate", 0.1),
            max_depth=params.pop("max_depth", 3),
            min_samples_split=params.pop("min_samples_split", 2),
            min_samples_leaf=params.pop("min_samples_leaf", 1),
            subsample=params.pop("subsample", 1.0),
            random_state=params.pop("random_state", RANDOM_STATE),
            **params,
        )

    if model_code == "XGB":
        return xgb.XGBClassifier(
            n_estimators=params.pop("n_estimators", 100),
            learning_rate=params.pop("learning_rate", 0.1),
            max_depth=params.pop("max_depth", 6),
            min_child_weight=params.pop("min_child_weight", 1),
            subsample=params.pop("subsample", 1.0),
            colsample_bytree=params.pop("colsample_bytree", 1.0),
            gamma=params.pop("gamma", 0.0),
            reg_alpha=params.pop("reg_alpha", 0.0),
            reg_lambda=params.pop("reg_lambda", 1.0),
            eval_metric="logloss",
            random_state=params.pop("random_state", RANDOM_STATE),
            n_jobs=N_JOBS_OPT,
            verbosity=0,
            **params,
        )

    if model_code == "LGBM":
        return lgb.LGBMClassifier(
            n_estimators=params.pop("n_estimators", 100),
            learning_rate=params.pop("learning_rate", 0.1),
            max_depth=params.pop("max_depth", -1),
            num_leaves=params.pop("num_leaves", 31),
            min_child_samples=params.pop("min_child_samples", 20),
            subsample=params.pop("subsample", 1.0),
            colsample_bytree=params.pop("colsample_bytree", 1.0),
            reg_alpha=params.pop("reg_alpha", 0.0),
            reg_lambda=params.pop("reg_lambda", 0.0),
            random_state=params.pop("random_state", RANDOM_STATE),
            verbose=-1,
            n_jobs=N_JOBS_OPT,
            **params,
        )

    if model_code == "CB":
        return CatBoostClassifier(
            iterations=params.pop("iterations", 100),
            learning_rate=params.pop("learning_rate", 0.1),
            depth=params.pop("depth", 6),
            l2_leaf_reg=params.pop("l2_leaf_reg", 3.0),
            border_count=params.pop("border_count", 128),
            bagging_temperature=params.pop("bagging_temperature", 0.0),
            random_seed=params.pop("random_state", RANDOM_STATE),
            verbose=0,
            allow_writing_files=False,
            **params,
        )

    if model_code == "SGD":
        return SGDClassifier(
            loss=params.pop("loss", "log_loss"),
            penalty=params.pop("penalty", "l2"),
            alpha=params.pop("alpha", 1e-4),
            l1_ratio=params.pop("l1_ratio", 0.15),
            max_iter=params.pop("max_iter", 2000),
            random_state=params.pop("random_state", RANDOM_STATE),
            **params,
        )

    raise ValueError(f"Unsupported model code: {model_code}")


def make_pipeline(model_code: str, params: Optional[Dict[str, Any]], scaler_name: str) -> Pipeline:
    steps = []
    scaler = get_scaler(scaler_name)
    if scaler is not None:
        steps.append(("scaler", scaler))
    steps.append(("model", build_model(model_code, params)))
    return Pipeline(steps)


def get_scores_and_probabilities(fitted_pipeline: Pipeline, X_input: pd.DataFrame) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    y_pred = fitted_pipeline.predict(X_input)
    y_proba = None
    y_score = None
    if hasattr(fitted_pipeline, "predict_proba"):
        y_proba = fitted_pipeline.predict_proba(X_input)[:, 1]
        y_score = y_proba.copy()
    elif hasattr(fitted_pipeline, "decision_function"):
        y_score = fitted_pipeline.decision_function(X_input)
    return y_pred, y_score, y_proba


def compute_metric_dict(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: Optional[np.ndarray] = None,
    y_proba: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    metrics = {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "AUC": np.nan,
        "LogLoss": np.nan,
    }

    if y_score is not None and len(np.unique(y_true)) > 1:
        try:
            metrics["AUC"] = float(roc_auc_score(y_true, y_score))
        except ValueError:
            metrics["AUC"] = np.nan

    if y_proba is not None and len(np.unique(y_true)) > 1:
        try:
            clipped = np.clip(y_proba, 1e-12, 1 - 1e-12)
            metrics["LogLoss"] = float(log_loss(y_true, clipped))
        except ValueError:
            metrics["LogLoss"] = np.nan

    return metrics


def bootstrap_metric_cis(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: Optional[np.ndarray] = None,
    y_proba: Optional[np.ndarray] = None,
    n_bootstrap: int = N_BOOTSTRAP,
) -> Dict[str, Tuple[float, float]]:
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(y_true))
    boot_samples = {metric: [] for metric in METRICS}

    for _ in range(n_bootstrap):
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)
        sampled_metrics = compute_metric_dict(
            y_true[sampled_idx],
            y_pred[sampled_idx],
            None if y_score is None else y_score[sampled_idx],
            None if y_proba is None else y_proba[sampled_idx],
        )
        for metric_name, metric_value in sampled_metrics.items():
            if not pd.isna(metric_value):
                boot_samples[metric_name].append(metric_value)

    intervals = {}
    for metric_name, values in boot_samples.items():
        if not values:
            intervals[metric_name] = (np.nan, np.nan)
        else:
            intervals[metric_name] = (
                float(np.percentile(values, 2.5)),
                float(np.percentile(values, 97.5)),
            )
    return intervals


def as_optional_array(values: np.ndarray) -> Optional[np.ndarray]:
    return None if np.isnan(values).all() else values


def evaluate_on_split(
    method_name: str,
    model_code: str,
    scaler_name: str,
    params: Optional[Dict[str, Any]],
    X_train_input: pd.DataFrame,
    y_train_input: pd.Series,
    X_test_input: pd.DataFrame,
    y_test_input: pd.Series,
) -> Tuple[pd.DataFrame, pd.DataFrame, Pipeline]:
    oof_predictions = np.zeros(len(y_train_input), dtype=int)
    oof_scores = np.full(len(y_train_input), np.nan, dtype=float)
    oof_proba = np.full(len(y_train_input), np.nan, dtype=float)
    fold_rows = []

    for fold_index, (fit_idx, val_idx) in enumerate(CV.split(X_train_input, y_train_input), start=1):
        pipeline = make_pipeline(model_code, params, scaler_name)
        X_fit = X_train_input.iloc[fit_idx]
        y_fit = y_train_input.iloc[fit_idx]
        X_val = X_train_input.iloc[val_idx]
        y_val = y_train_input.iloc[val_idx]

        pipeline.fit(X_fit, y_fit)
        y_pred_fold, y_score_fold, y_proba_fold = get_scores_and_probabilities(pipeline, X_val)

        oof_predictions[val_idx] = y_pred_fold
        if y_score_fold is not None:
            oof_scores[val_idx] = y_score_fold
        if y_proba_fold is not None:
            oof_proba[val_idx] = y_proba_fold

        fold_rows.append(
            {
                "Method": method_name,
                "Model": model_code,
                "Optimal_Scaler": scaler_name,
                "Fold": fold_index,
                **compute_metric_dict(
                    y_val.to_numpy(),
                    y_pred_fold,
                    None if y_score_fold is None else np.asarray(y_score_fold),
                    None if y_proba_fold is None else np.asarray(y_proba_fold),
                ),
            }
        )

    final_pipeline = make_pipeline(model_code, params, scaler_name)
    final_pipeline.fit(X_train_input, y_train_input)
    y_pred_test, y_score_test, y_proba_test = get_scores_and_probabilities(final_pipeline, X_test_input)

    train_metrics = compute_metric_dict(
        y_train_input.to_numpy(),
        oof_predictions,
        as_optional_array(oof_scores),
        as_optional_array(oof_proba),
    )
    test_metrics = compute_metric_dict(
        y_test_input.to_numpy(),
        y_pred_test,
        None if y_score_test is None else np.asarray(y_score_test),
        None if y_proba_test is None else np.asarray(y_proba_test),
    )

    train_ci = bootstrap_metric_cis(
        y_train_input.to_numpy(),
        oof_predictions,
        as_optional_array(oof_scores),
        as_optional_array(oof_proba),
    )
    test_ci = bootstrap_metric_cis(
        y_test_input.to_numpy(),
        y_pred_test,
        None if y_score_test is None else np.asarray(y_score_test),
        None if y_proba_test is None else np.asarray(y_proba_test),
    )

    summary_rows = []
    for split_name, metric_values, ci_values in [("Train_OOF", train_metrics, train_ci), ("Test", test_metrics, test_ci)]:
        row = {
            "Method": method_name,
            "Model": model_code,
            "Optimal_Scaler": scaler_name,
            "Split": split_name,
        }
        row.update(metric_values)
        for metric_name in METRICS:
            row[f"{metric_name}_CI_Lower"] = ci_values[metric_name][0]
            row[f"{metric_name}_CI_Upper"] = ci_values[metric_name][1]
        summary_rows.append(row)

    return pd.DataFrame(fold_rows), pd.DataFrame(summary_rows), final_pipeline


def sample_params_tpe(trial: optuna.trial.Trial, model_code: str) -> Dict[str, Any]:
    sampled = {}
    for param_name, spec in SEARCH_SPACES[model_code].items():
        if spec["type"] == "categorical":
            sampled[param_name] = trial.suggest_categorical(param_name, spec["choices"])
        elif spec["type"] == "int":
            sampled[param_name] = trial.suggest_int(param_name, spec["range"][0], spec["range"][1], log=spec["log"])
        elif spec["type"] == "float":
            sampled[param_name] = trial.suggest_float(param_name, spec["range"][0], spec["range"][1], log=spec["log"])
        else:
            raise ValueError(f"Unsupported parameter type: {spec['type']}")

    if model_code == "LR":
        sampled.update({"solver": "saga", "max_iter": 2000, "random_state": RANDOM_STATE})
    elif model_code == "SGD":
        sampled.update({"max_iter": 2000, "random_state": RANDOM_STATE})
    elif model_code == "LinearSVC":
        sampled.update({"dual": "auto", "max_iter": 5000, "random_state": RANDOM_STATE})
    elif model_code in {"RF", "AdaBoost", "GB", "XGB", "LGBM", "CB"}:
        sampled.update({"random_state": RANDOM_STATE})
    return sampled


def decode_gwo_position(position: np.ndarray, model_code: str) -> Dict[str, Any]:
    params = {}
    for dim_index, (param_name, spec) in enumerate(SEARCH_SPACES[model_code].items()):
        raw_value = float(np.clip(position[dim_index], 0.0, 1.0))
        if spec["type"] == "categorical":
            choices = spec["choices"]
            choice_index = int(round(raw_value * (len(choices) - 1)))
            params[param_name] = choices[min(max(choice_index, 0), len(choices) - 1)]
        elif spec["type"] == "int":
            low, high = spec["range"]
            params[param_name] = int(round(low + raw_value * (high - low)))
        elif spec["type"] == "float":
            low, high = spec["range"]
            if spec["log"]:
                params[param_name] = float(np.exp(np.log(low) + raw_value * (np.log(high) - np.log(low))))
            else:
                params[param_name] = float(low + raw_value * (high - low))
        else:
            raise ValueError(f"Unsupported parameter type: {spec['type']}")

    if model_code == "LR":
        params.update({"solver": "saga", "max_iter": 2000, "random_state": RANDOM_STATE})
    elif model_code == "SGD":
        params.update({"max_iter": 2000, "random_state": RANDOM_STATE})
    elif model_code == "LinearSVC":
        params.update({"dual": "auto", "max_iter": 5000, "random_state": RANDOM_STATE})
    elif model_code in {"RF", "AdaBoost", "GB", "XGB", "LGBM", "CB"}:
        params.update({"random_state": RANDOM_STATE})
    return params


def compare_search_spaces() -> pd.DataFrame:
    rows = []
    for model_code, space in SEARCH_SPACES.items():
        for param_name, spec in space.items():
            rows.append(
                {
                    "Model": model_code,
                    "Hyperparameter": param_name,
                    "Type": spec["type"],
                    "Lower_Bound": None if spec["type"] == "categorical" else spec["range"][0],
                    "Upper_Bound": None if spec["type"] == "categorical" else spec["range"][1],
                    "Choices": None if spec["type"] != "categorical" else "|".join([str(choice) for choice in spec["choices"]]),
                    "Log_Scale": None if spec["type"] == "categorical" else spec["log"],
                }
            )
    return pd.DataFrame(rows)


search_space_df = compare_search_spaces()
search_space_df.to_csv(RESULTS_DIR / "inline_search_space.csv", index=False)
display(search_space_df.head(20))

,Model,Hyperparameter,Type,Lower_Bound,Upper_Bound,Choices,Log_Scale
0,LR,C,float,1.000000e-04,100.000000,None,True
1,LR,penalty,categorical,NaN,NaN,l1|l2,None
2,SGD,loss,categorical,NaN,NaN,log_loss|modified_huber,None
3,SGD,penalty,categorical,NaN,NaN,l1|l2|elasticnet,None
4,SGD,alpha,float,1.000000e-06,0.100000,None,True
5,SGD,l1_ratio,float,0.000000e+00,1.000000,None,False
6,LDA,solver,categorical,NaN,NaN,svd|lsqr|eigen,None
7,QDA,reg_param,float,0.000000e+00,1.000000,None,False
8,NB,var_smoothing,float,1.000000e-12,0.000001,None,True
9,KNN,n_neighbors,int,1.000000e+00,50.000000,None,False


## baseline evaluation

In [12]:
baseline_cv_rows = []
baseline_summary_rows = []
baseline_best_scalers = []

print("\n" + "="*80)
print("BASELINE EVALUATION - Scaler Sweep (14 models × 6 scalers)")
print("="*80)
baseline_total_start = time.perf_counter()

for model_idx, model_code in enumerate(MODEL_ORDER, start=1):
    model_start = time.perf_counter()
    print(f"\n[{model_idx:2d}/{len(MODEL_ORDER)}] {model_code:10s} | Scaler Sweep (6 scalers)")
    
    scaler_scores = []
    for scaler_idx, scaler_name in enumerate(SCALER_FACTORIES, start=1):
        with ResourceMonitor(f"  {scaler_name:12s}") as rm:
            pipeline = make_pipeline(model_code, None, scaler_name)
            cv_out = cross_validate(
                pipeline,
                X_train,
                y_train,
                cv=CV,
                scoring="accuracy",
                n_jobs=N_JOBS_CV,
                return_train_score=False,
            )
        fold_scores = cv_out["test_score"]
        scaler_scores.append(
            {
                "Model": model_code,
                "Scaler": scaler_name,
                "Mean": float(np.mean(fold_scores)),
                "Std": float(np.std(fold_scores, ddof=0)),
                **{f"Fold_{index}": float(score) for index, score in enumerate(fold_scores, start=1)},
            }
        )

    scaler_scores_df = pd.DataFrame(scaler_scores).sort_values(["Mean", "Std"], ascending=[False, True])
    best_row = scaler_scores_df.iloc[0].to_dict()
    best_scaler = best_row["Scaler"]
    baseline_best_scalers.append(
        {
            "Model": model_code,
            "Optimal_Scaler": best_scaler,
            "CV_Mean": best_row["Mean"],
            "CV_Std": best_row["Std"],
        }
    )
    baseline_cv_rows.append(scaler_scores_df)

    with ResourceMonitor(f"  Eval {model_code}") as rm:
        fold_df, summary_df, _ = evaluate_on_split(
            method_name="Baseline",
            model_code=model_code,
            scaler_name=best_scaler,
            params=None,
            X_train_input=X_train,
            y_train_input=y_train,
            X_test_input=X_test,
            y_test_input=y_test,
        )
    baseline_summary_rows.append(summary_df)
    
    model_time = time.perf_counter() - model_start
    print(f"       ✓ {model_code} best scaler: {best_scaler:12s} | CV: {best_row['Mean']:.4f} | Time: {model_time:6.1f}s")

baseline_total_time = time.perf_counter() - baseline_total_start

baseline_scaler_grid = pd.concat(baseline_cv_rows, ignore_index=True)
baseline_best_scalers_df = pd.DataFrame(baseline_best_scalers).sort_values("Model")
baseline_results_df = pd.concat(baseline_summary_rows, ignore_index=True).sort_values(["Model", "Split"])

baseline_train_results_df = baseline_results_df[baseline_results_df["Split"] == "Train_OOF"].reset_index(drop=True)
baseline_test_results_df = baseline_results_df[baseline_results_df["Split"] == "Test"].reset_index(drop=True)

baseline_scaler_grid.to_csv(RESULTS_DIR / "baseline_scaler_grid.csv", index=False)
baseline_best_scalers_df.to_csv(RESULTS_DIR / "baseline_optimal_scalers.csv", index=False)
baseline_train_results_df.to_csv(RESULTS_DIR / "baseline_train_results_with_ci.csv", index=False)
baseline_test_results_df.to_csv(RESULTS_DIR / "baseline_test_results_with_ci.csv", index=False)

print("\n" + "="*80)
print(f"Baseline evaluation completed in {baseline_total_time:.1f}s ({baseline_total_time/60:.2f} min)")
print("="*80)
display(baseline_best_scalers_df)
display(baseline_test_results_df[["Method", "Model", "Optimal_Scaler", *METRICS]].sort_values("Accuracy", ascending=False))


BASELINE EVALUATION - Scaler Sweep (14 models × 6 scalers)

[ 1/14] LR         | Scaler Sweep (6 scalers)
  [  Raw         ] Time:    8.30s | Memory Δ:    +5.9MB | Peak:   255.6MB
  [  Standard    ] Time:    0.91s | Memory Δ:    -4.9MB | Peak:   250.8MB
  [  MinMax      ] Time:    1.74s | Memory Δ:    +0.0MB | Peak:   250.8MB
  [  Robust      ] Time:    1.43s | Memory Δ:    +5.8MB | Peak:   256.5MB
  [  Power       ] Time:    5.09s | Memory Δ:    +0.3MB | Peak:   256.8MB
  [  Quantile    ] Time:    1.72s | Memory Δ:    -5.5MB | Peak:   251.3MB
  [  Eval LR] Time:  110.95s | Memory Δ:    +1.1MB | Peak:   252.4MB
       ✓ LR best scaler: Power        | CV: 0.8953 | Time:  130.8s

[ 2/14] LDA        | Scaler Sweep (6 scalers)
  [  Raw         ] Time:    0.33s | Memory Δ:    +6.2MB | Peak:   258.6MB
  [  Standard    ] Time:    0.33s | Memory Δ:    +1.0MB | Peak:   259.6MB
  [  MinMax      ] Time:    0.29s | Memory Δ:    +0.0MB | Peak:   259.6MB
  [  Robust      ] Time:    0.44s | Memory Δ

,Model,Optimal_Scaler,CV_Mean,CV_Std
7,AdaBoost,Raw,0.887562,0.007040
12,CB,MinMax,0.939271,0.001707
9,GB,Raw,0.922583,0.003934
4,KNN,Raw,0.932667,0.002626
1,LDA,Power,0.895312,0.002253
11,LGBM,Raw,0.967750,0.001915
0,LR,Power,0.895271,0.003145
5,LinearSVC,Power,0.897000,0.002657
3,NB,Power,0.843042,0.001900
2,QDA,Power,0.892437,0.001838


,Method,Model,Optimal_Scaler,Accuracy,Precision,Recall,F1,AUC,LogLoss,Kappa,MCC
12,Baseline,SVM,Standard,0.982333,0.984635,0.987722,0.986176,0.998714,0.043883,0.961707,0.961716
13,Baseline,XGB,MinMax,0.972083,0.971896,0.984718,0.978265,0.997072,0.104082,0.939257,0.939415
5,Baseline,LGBM,Raw,0.971083,0.971488,0.983542,0.977478,0.996897,0.106920,0.937100,0.937239
10,Baseline,RF,MinMax,0.969000,0.967402,0.984587,0.975919,0.996245,0.138566,0.932430,0.932715
1,Baseline,CB,MinMax,0.942583,0.945745,0.965387,0.955465,0.988919,0.171676,0.874706,0.875072
3,Baseline,KNN,Raw,0.939417,0.946514,0.959248,0.952838,0.982171,0.245021,0.868166,0.868318
2,Baseline,GB,Raw,0.926250,0.928490,0.958203,0.943112,0.982510,0.204890,0.838366,0.839206
7,Baseline,LinearSVC,Power,0.905167,0.917928,0.934953,0.926362,0.968560,NaN,0.793232,0.793496
4,Baseline,LDA,Power,0.903500,0.917287,0.932863,0.925010,0.967520,0.234272,0.789725,0.789945
6,Baseline,LR,Power,0.903083,0.915525,0.934300,0.924817,0.968221,0.220371,0.788531,0.788852


## 🔁 Recovery Cells
Run these cells **after a kernel restart** to continue optimization without rerunning expensive feature selection or baseline sweep.

| Cell | Restores |
|---|---|
| Recovery 1 | `df`, `y`, `X_raw`, `X_engineered`, `X_train_full`, `X_test_full`, `y_train`, `y_test`, `selected_features`, `X_train`, `X_test` |
| Recovery 2 | `baseline_best_scalers_df` (loaded from `baseline_optimal_scalers.csv`) |
| Optimization shared helpers | all utility functions required by TPE/GWO resume, including model/scaler builders, samplers, metrics, and `evaluate_on_split` |

After a restart, run in this order:
1. setup/imports
2. system resource setup
3. Recovery 1
4. Recovery 2
5. optimization shared helpers
6. the TPE or GWO model cell you want to continue


In [30]:
# ============================================================
# RECOVERY 1 — Data split + selected features (no recomputation)
# Prerequisite: setup cell (imports + constants)
# ============================================================

def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    tau_cols = ["tau1", "tau2", "tau3", "tau4"]
    g_cols   = ["g1",  "g2",  "g3",  "g4"]
    X["tau_mean"]    = X[tau_cols].mean(axis=1)
    X["g_sum"]       = X[g_cols].sum(axis=1)
    X["tau1_g1"]     = X["tau1"] * X["g1"]
    X["tau2_g2"]     = X["tau2"] * X["g2"]
    X["tau3_g3"]     = X["tau3"] * X["g3"]
    X["tau4_g4"]     = X["tau4"] * X["g4"]
    X["g_tau_ratio"] = X["g_sum"] / (X["tau_mean"] + 1e-5)
    return X

df           = pd.read_csv(DATA_PATH)
y            = (df[TARGET_COLUMN] == "unstable").astype(int)
X_raw        = df.drop(columns=[TARGET_COLUMN, LEAKAGE_COLUMN])
X_engineered = engineer_features(X_raw)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_engineered, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Load from saved artifact — skips expensive consensus_feature_selection()
_sf_path = RESULTS_DIR / "selected_features.csv"
selected_features = pd.read_csv(_sf_path)["feature"].dropna().astype(str).tolist()

_missing = [f for f in selected_features if f not in X_train_full.columns]
if _missing:
    raise ValueError(f"[Recovery] Features missing from engineered matrix: {_missing}")

X_train = X_train_full[selected_features].copy()
X_test  = X_test_full[selected_features].copy()

assert len(X_train) == 48000 and len(X_test) == 12000
print(f"[Recovery 1] train={X_train.shape}  test={X_test.shape}  features={len(selected_features)}")


[Recovery 1] train=(48000, 13)  test=(12000, 13)  features=13


In [31]:
# ============================================================
# RECOVERY 2 — Restore only scaler map needed by optimization
# Prerequisite: setup cell
# ============================================================

baseline_best_scalers_df = pd.read_csv(RESULTS_DIR / "baseline_optimal_scalers.csv")

required_cols = {"Model", "Optimal_Scaler"}
missing_cols = required_cols - set(baseline_best_scalers_df.columns)
if missing_cols:
    raise KeyError(f"baseline_optimal_scalers.csv missing required columns: {sorted(missing_cols)}")

baseline_best_scalers_df = baseline_best_scalers_df.sort_values("Model").reset_index(drop=True)

print(f"[Recovery 2] Restored baseline_best_scalers_df with {len(baseline_best_scalers_df)} models")
display(baseline_best_scalers_df)


[Recovery 2] Restored baseline_best_scalers_df with 14 models


,Model,Optimal_Scaler,CV_Mean,CV_Std
0,AdaBoost,Raw,0.887562,0.007040
1,CB,MinMax,0.939271,0.001707
2,GB,Raw,0.922583,0.003934
3,KNN,Raw,0.932667,0.002626
4,LDA,Power,0.895312,0.002253
5,LGBM,Raw,0.967750,0.001915
6,LR,Power,0.895271,0.003145
7,LinearSVC,Power,0.897000,0.002657
8,NB,Power,0.843042,0.001900
9,QDA,Power,0.892437,0.001838


## optimization shared helpers

In [32]:
# Recovery shim: define optimization dependencies if utility cell was not run.

if "get_scaler" not in globals():
    def get_scaler(scaler_name: str):
        factory = SCALER_FACTORIES[scaler_name]
        return None if factory is None or scaler_name == "Raw" else factory()

if "build_model" not in globals():
    def build_model(model_code: str, params: Optional[Dict[str, Any]] = None):
        params = dict(params or {})

        if model_code == "LR":
            penalty = params.pop("penalty", "l2")
            solver = params.pop("solver", "saga")
            if penalty == "l1" and solver not in {"liblinear", "saga"}:
                solver = "saga"
            return LogisticRegression(
                C=params.pop("C", 1.0),
                penalty=penalty,
                solver=solver,
                max_iter=params.pop("max_iter", 2000),
                random_state=params.pop("random_state", RANDOM_STATE),
                **params,
            )

        if model_code == "LDA":
            solver = params.pop("solver", "svd")
            shrinkage = params.pop("shrinkage", None)
            if solver == "svd":
                return LinearDiscriminantAnalysis(solver=solver, **params)
            return LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage, **params)

        if model_code == "QDA":
            return QuadraticDiscriminantAnalysis(reg_param=params.pop("reg_param", 0.0), **params)

        if model_code == "NB":
            return GaussianNB(var_smoothing=params.pop("var_smoothing", 1e-9), **params)

        if model_code == "KNN":
            return KNeighborsClassifier(
                n_neighbors=params.pop("n_neighbors", 5),
                weights=params.pop("weights", "uniform"),
                metric=params.pop("metric", "minkowski"),
                p=params.pop("p", 2),
                **params,
            )

        if model_code == "LinearSVC":
            return LinearSVC(
                C=params.pop("C", 1.0),
                loss=params.pop("loss", "squared_hinge"),
                dual=params.pop("dual", "auto"),
                max_iter=params.pop("max_iter", 5000),
                random_state=params.pop("random_state", RANDOM_STATE),
                **params,
            )

        if model_code == "SVM":
            return SVC(
                kernel=params.pop("kernel", "rbf"),
                C=params.pop("C", 1.0),
                gamma=params.pop("gamma", "scale"),
                degree=params.pop("degree", 3),
                coef0=params.pop("coef0", 0.0),
                probability=True,
                random_state=RANDOM_STATE,
                **params,
            )

        if model_code == "AdaBoost":
            algorithm = params.pop("algorithm", "SAMME")
            if algorithm == "SAMME.R":
                algorithm = "SAMME"
            return AdaBoostClassifier(
                n_estimators=params.pop("n_estimators", 50),
                learning_rate=params.pop("learning_rate", 1.0),
                algorithm=algorithm,
                random_state=params.pop("random_state", RANDOM_STATE),
                **params,
            )

        if model_code == "RF":
            max_features = params.pop("max_features", "sqrt")
            if max_features == "None":
                max_features = None
            return RandomForestClassifier(
                n_estimators=params.pop("n_estimators", 100),
                max_depth=params.pop("max_depth", None),
                min_samples_split=params.pop("min_samples_split", 2),
                min_samples_leaf=params.pop("min_samples_leaf", 1),
                max_features=max_features,
                random_state=params.pop("random_state", RANDOM_STATE),
                n_jobs=N_JOBS_OPT,
                **params,
            )

        if model_code == "GB":
            return GradientBoostingClassifier(
                n_estimators=params.pop("n_estimators", 100),
                learning_rate=params.pop("learning_rate", 0.1),
                max_depth=params.pop("max_depth", 3),
                min_samples_split=params.pop("min_samples_split", 2),
                min_samples_leaf=params.pop("min_samples_leaf", 1),
                subsample=params.pop("subsample", 1.0),
                random_state=params.pop("random_state", RANDOM_STATE),
                **params,
            )

        if model_code == "XGB":
            return xgb.XGBClassifier(
                n_estimators=params.pop("n_estimators", 100),
                learning_rate=params.pop("learning_rate", 0.1),
                max_depth=params.pop("max_depth", 6),
                min_child_weight=params.pop("min_child_weight", 1),
                subsample=params.pop("subsample", 1.0),
                colsample_bytree=params.pop("colsample_bytree", 1.0),
                gamma=params.pop("gamma", 0.0),
                reg_alpha=params.pop("reg_alpha", 0.0),
                reg_lambda=params.pop("reg_lambda", 1.0),
                eval_metric="logloss",
                random_state=params.pop("random_state", RANDOM_STATE),
                n_jobs=N_JOBS_OPT,
                verbosity=0,
                **params,
            )

        if model_code == "LGBM":
            return lgb.LGBMClassifier(
                n_estimators=params.pop("n_estimators", 100),
                learning_rate=params.pop("learning_rate", 0.1),
                max_depth=params.pop("max_depth", -1),
                num_leaves=params.pop("num_leaves", 31),
                min_child_samples=params.pop("min_child_samples", 20),
                subsample=params.pop("subsample", 1.0),
                colsample_bytree=params.pop("colsample_bytree", 1.0),
                reg_alpha=params.pop("reg_alpha", 0.0),
                reg_lambda=params.pop("reg_lambda", 0.0),
                random_state=params.pop("random_state", RANDOM_STATE),
                verbose=-1,
                n_jobs=N_JOBS_OPT,
                **params,
            )

        if model_code == "CB":
            return CatBoostClassifier(
                iterations=params.pop("iterations", 100),
                learning_rate=params.pop("learning_rate", 0.1),
                depth=params.pop("depth", 6),
                l2_leaf_reg=params.pop("l2_leaf_reg", 3.0),
                border_count=params.pop("border_count", 128),
                bagging_temperature=params.pop("bagging_temperature", 0.0),
                random_seed=params.pop("random_state", RANDOM_STATE),
                verbose=0,
                allow_writing_files=False,
                **params,
            )

        if model_code == "SGD":
            return SGDClassifier(
                loss=params.pop("loss", "log_loss"),
                penalty=params.pop("penalty", "l2"),
                alpha=params.pop("alpha", 1e-4),
                l1_ratio=params.pop("l1_ratio", 0.15),
                max_iter=params.pop("max_iter", 2000),
                random_state=params.pop("random_state", RANDOM_STATE),
                **params,
            )

        raise ValueError(f"Unsupported model code: {model_code}")

if "make_pipeline" not in globals():
    def make_pipeline(model_code: str, params: Optional[Dict[str, Any]], scaler_name: str) -> Pipeline:
        steps = []
        scaler = get_scaler(scaler_name)
        if scaler is not None:
            steps.append(("scaler", scaler))
        steps.append(("model", build_model(model_code, params)))
        return Pipeline(steps)

if "get_scores_and_probabilities" not in globals():
    def get_scores_and_probabilities(fitted_pipeline: Pipeline, X_input: pd.DataFrame) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
        y_pred = fitted_pipeline.predict(X_input)
        y_proba = None
        y_score = None
        if hasattr(fitted_pipeline, "predict_proba"):
            y_proba = fitted_pipeline.predict_proba(X_input)[:, 1]
            y_score = y_proba.copy()
        elif hasattr(fitted_pipeline, "decision_function"):
            y_score = fitted_pipeline.decision_function(X_input)
        return y_pred, y_score, y_proba

if "compute_metric_dict" not in globals():
    def compute_metric_dict(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_score: Optional[np.ndarray] = None,
        y_proba: Optional[np.ndarray] = None,
    ) -> Dict[str, float]:
        metrics = {
            "Accuracy": float(accuracy_score(y_true, y_pred)),
            "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "F1": float(f1_score(y_true, y_pred, zero_division=0)),
            "Kappa": float(cohen_kappa_score(y_true, y_pred)),
            "MCC": float(matthews_corrcoef(y_true, y_pred)),
            "AUC": np.nan,
            "LogLoss": np.nan,
        }

        if y_score is not None and len(np.unique(y_true)) > 1:
            try:
                metrics["AUC"] = float(roc_auc_score(y_true, y_score))
            except ValueError:
                metrics["AUC"] = np.nan

        if y_proba is not None and len(np.unique(y_true)) > 1:
            try:
                clipped = np.clip(y_proba, 1e-12, 1 - 1e-12)
                metrics["LogLoss"] = float(log_loss(y_true, clipped))
            except ValueError:
                metrics["LogLoss"] = np.nan

        return metrics

if "bootstrap_metric_cis" not in globals():
    def bootstrap_metric_cis(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_score: Optional[np.ndarray] = None,
        y_proba: Optional[np.ndarray] = None,
        n_bootstrap: int = N_BOOTSTRAP,
    ) -> Dict[str, Tuple[float, float]]:
        rng = np.random.default_rng(RANDOM_STATE)
        indices = np.arange(len(y_true))
        boot_samples = {metric: [] for metric in METRICS}

        for _ in range(n_bootstrap):
            sampled_idx = rng.choice(indices, size=len(indices), replace=True)
            sampled_metrics = compute_metric_dict(
                y_true[sampled_idx],
                y_pred[sampled_idx],
                None if y_score is None else y_score[sampled_idx],
                None if y_proba is None else y_proba[sampled_idx],
            )
            for metric_name, metric_value in sampled_metrics.items():
                if not pd.isna(metric_value):
                    boot_samples[metric_name].append(metric_value)

        intervals = {}
        for metric_name, values in boot_samples.items():
            if not values:
                intervals[metric_name] = (np.nan, np.nan)
            else:
                intervals[metric_name] = (
                    float(np.percentile(values, 2.5)),
                    float(np.percentile(values, 97.5)),
                )
        return intervals

if "as_optional_array" not in globals():
    def as_optional_array(values: np.ndarray) -> Optional[np.ndarray]:
        return None if np.isnan(values).all() else values

if "evaluate_on_split" not in globals():
    def evaluate_on_split(
        method_name: str,
        model_code: str,
        scaler_name: str,
        params: Optional[Dict[str, Any]],
        X_train_input: pd.DataFrame,
        y_train_input: pd.Series,
        X_test_input: pd.DataFrame,
        y_test_input: pd.Series,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Pipeline]:
        oof_predictions = np.zeros(len(y_train_input), dtype=int)
        oof_scores = np.full(len(y_train_input), np.nan, dtype=float)
        oof_proba = np.full(len(y_train_input), np.nan, dtype=float)
        fold_rows = []

        for fold_index, (fit_idx, val_idx) in enumerate(CV.split(X_train_input, y_train_input), start=1):
            pipeline = make_pipeline(model_code, params, scaler_name)
            X_fit = X_train_input.iloc[fit_idx]
            y_fit = y_train_input.iloc[fit_idx]
            X_val = X_train_input.iloc[val_idx]
            y_val = y_train_input.iloc[val_idx]

            pipeline.fit(X_fit, y_fit)
            y_pred_fold, y_score_fold, y_proba_fold = get_scores_and_probabilities(pipeline, X_val)

            oof_predictions[val_idx] = y_pred_fold
            if y_score_fold is not None:
                oof_scores[val_idx] = y_score_fold
            if y_proba_fold is not None:
                oof_proba[val_idx] = y_proba_fold

            fold_rows.append(
                {
                    "Method": method_name,
                    "Model": model_code,
                    "Optimal_Scaler": scaler_name,
                    "Fold": fold_index,
                    **compute_metric_dict(
                        y_val.to_numpy(),
                        y_pred_fold,
                        None if y_score_fold is None else np.asarray(y_score_fold),
                        None if y_proba_fold is None else np.asarray(y_proba_fold),
                    ),
                }
            )

        final_pipeline = make_pipeline(model_code, params, scaler_name)
        final_pipeline.fit(X_train_input, y_train_input)
        y_pred_test, y_score_test, y_proba_test = get_scores_and_probabilities(final_pipeline, X_test_input)

        train_metrics = compute_metric_dict(
            y_train_input.to_numpy(),
            oof_predictions,
            as_optional_array(oof_scores),
            as_optional_array(oof_proba),
        )
        test_metrics = compute_metric_dict(
            y_test_input.to_numpy(),
            y_pred_test,
            None if y_score_test is None else np.asarray(y_score_test),
            None if y_proba_test is None else np.asarray(y_proba_test),
        )

        train_ci = bootstrap_metric_cis(
            y_train_input.to_numpy(),
            oof_predictions,
            as_optional_array(oof_scores),
            as_optional_array(oof_proba),
        )
        test_ci = bootstrap_metric_cis(
            y_test_input.to_numpy(),
            y_pred_test,
            None if y_score_test is None else np.asarray(y_score_test),
            None if y_proba_test is None else np.asarray(y_proba_test),
        )

        summary_rows = []
        for split_name, metric_values, ci_values in [("Train_OOF", train_metrics, train_ci), ("Test", test_metrics, test_ci)]:
            row = {
                "Method": method_name,
                "Model": model_code,
                "Optimal_Scaler": scaler_name,
                "Split": split_name,
            }
            row.update(metric_values)
            for metric_name in METRICS:
                row[f"{metric_name}_CI_Lower"] = ci_values[metric_name][0]
                row[f"{metric_name}_CI_Upper"] = ci_values[metric_name][1]
            summary_rows.append(row)

        return pd.DataFrame(fold_rows), pd.DataFrame(summary_rows), final_pipeline

if "sample_params_tpe" not in globals():
    def sample_params_tpe(trial: optuna.trial.Trial, model_code: str) -> Dict[str, Any]:
        sampled = {}
        for param_name, spec in SEARCH_SPACES[model_code].items():
            if spec["type"] == "categorical":
                sampled[param_name] = trial.suggest_categorical(param_name, spec["choices"])
            elif spec["type"] == "int":
                sampled[param_name] = trial.suggest_int(param_name, spec["range"][0], spec["range"][1], log=spec["log"])
            elif spec["type"] == "float":
                sampled[param_name] = trial.suggest_float(param_name, spec["range"][0], spec["range"][1], log=spec["log"])
            else:
                raise ValueError(f"Unsupported parameter type: {spec['type']}")

        if model_code == "LR":
            sampled.update({"solver": "saga", "max_iter": 2000, "random_state": RANDOM_STATE})
        elif model_code == "SGD":
            sampled.update({"max_iter": 2000, "random_state": RANDOM_STATE})
        elif model_code == "LinearSVC":
            sampled.update({"dual": "auto", "max_iter": 5000, "random_state": RANDOM_STATE})
        elif model_code in {"RF", "AdaBoost", "GB", "XGB", "LGBM", "CB"}:
            sampled.update({"random_state": RANDOM_STATE})
        return sampled

if "decode_gwo_position" not in globals():
    def decode_gwo_position(position: np.ndarray, model_code: str) -> Dict[str, Any]:
        params = {}
        for dim_index, (param_name, spec) in enumerate(SEARCH_SPACES[model_code].items()):
            raw_value = float(np.clip(position[dim_index], 0.0, 1.0))
            if spec["type"] == "categorical":
                choices = spec["choices"]
                choice_index = int(round(raw_value * (len(choices) - 1)))
                params[param_name] = choices[min(max(choice_index, 0), len(choices) - 1)]
            elif spec["type"] == "int":
                low, high = spec["range"]
                params[param_name] = int(round(low + raw_value * (high - low)))
            elif spec["type"] == "float":
                low, high = spec["range"]
                if spec["log"]:
                    params[param_name] = float(np.exp(np.log(low) + raw_value * (np.log(high) - np.log(low))))
                else:
                    params[param_name] = float(low + raw_value * (high - low))
            else:
                raise ValueError(f"Unsupported parameter type: {spec['type']}")

        if model_code == "LR":
            params.update({"solver": "saga", "max_iter": 2000, "random_state": RANDOM_STATE})
        elif model_code == "SGD":
            params.update({"max_iter": 2000, "random_state": RANDOM_STATE})
        elif model_code == "LinearSVC":
            params.update({"dual": "auto", "max_iter": 5000, "random_state": RANDOM_STATE})
        elif model_code in {"RF", "AdaBoost", "GB", "XGB", "LGBM", "CB"}:
            params.update({"random_state": RANDOM_STATE})
        return params

print("Optimization recovery shim ready: model/scaler builders, samplers, metrics, and evaluate_on_split restored.")

Optimization recovery shim ready: model/scaler builders, samplers, metrics, and evaluate_on_split restored.


In [33]:
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = RESULTS_DIR / "optimization_progress.json"

FIXED_N_JOBS_CV = N_JOBS_CV
FIXED_N_JOBS_OPT = N_JOBS_OPT


def load_progress_log() -> Dict[str, Dict[str, str]]:
    if PROGRESS_LOG.exists():
        return json.loads(PROGRESS_LOG.read_text())
    return {}


def save_progress_log(log: Dict[str, Dict[str, str]]) -> None:
    with open(PROGRESS_LOG, "w", encoding="utf-8") as handle:
        json.dump(log, handle, indent=2)


def refresh_memory_snapshot() -> Dict[str, float]:
    vm = psutil.virtual_memory()
    return {
        "available_gb": float(vm.available / (1024**3)),
        "used_percent": float(vm.percent),
        "process_mb": float(psutil.Process(os.getpid()).memory_info().rss / (1024**2)),
    }


def enforce_fair_resource_policy() -> None:
    global N_JOBS_CV, N_JOBS_OPT
    N_JOBS_CV = FIXED_N_JOBS_CV
    N_JOBS_OPT = FIXED_N_JOBS_OPT


def collect_runtime_garbage() -> None:
    gc.collect()


def method_paths(method_name: str) -> Dict[str, Path]:
    prefix = method_name.lower()
    return {
        "best": RESULTS_DIR / f"{prefix}_best_params.csv",
        "history": RESULTS_DIR / f"{prefix}_trials_full.csv",
        "train": RESULTS_DIR / f"{prefix}_train_results_with_ci.csv",
        "test": RESULTS_DIR / f"{prefix}_test_results_with_ci.csv",
        "timing": RESULTS_DIR / f"{prefix}_timing_per_model.csv",
    }


def load_existing_method_outputs(method_name: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths = method_paths(method_name)
    best_df = pd.read_csv(paths["best"]) if paths["best"].exists() else pd.DataFrame()
    history_df = pd.read_csv(paths["history"]) if paths["history"].exists() else pd.DataFrame()
    timing_df = pd.read_csv(paths["timing"]) if paths["timing"].exists() else pd.DataFrame()

    result_frames = []
    if paths["train"].exists():
        result_frames.append(pd.read_csv(paths["train"]))
    if paths["test"].exists():
        result_frames.append(pd.read_csv(paths["test"]))
    results_df = pd.concat(result_frames, ignore_index=True) if result_frames else pd.DataFrame()
    return best_df, history_df, results_df, timing_df


def deduplicate_rows(df: pd.DataFrame, subset: List[str]) -> pd.DataFrame:
    if df.empty:
        return df
    available_subset = [column for column in subset if column in df.columns]
    if not available_subset:
        return df
    return df.drop_duplicates(subset=available_subset, keep="last").reset_index(drop=True)


def save_model_checkpoint(method_name: str, model_code: str, payload: Dict[str, Any]) -> None:
    checkpoint_file = CHECKPOINT_DIR / f"{method_name.lower()}_{model_code}.json"
    with open(checkpoint_file, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)


def export_method_outputs(
    method_name: str,
    best_df: pd.DataFrame,
    history_df: pd.DataFrame,
    results_df: pd.DataFrame,
    timing_df: pd.DataFrame,
    print_status: bool = False,
) -> None:
    paths = method_paths(method_name)
    if not best_df.empty:
        best_df.sort_values("Model").to_csv(paths["best"], index=False)
    if not history_df.empty:
        history_df.to_csv(paths["history"], index=False)
    if not results_df.empty:
        results_df[results_df["Split"] == "Train_OOF"].to_csv(paths["train"], index=False)
        results_df[results_df["Split"] == "Test"].to_csv(paths["test"], index=False)
    if not timing_df.empty:
        timing_df.sort_values("Model").to_csv(paths["timing"], index=False)
    if print_status:
        print(f"       Exported {method_name} artifacts")


def objective_from_params(model_code: str, scaler_name: str, params: Dict[str, Any]) -> float:
    enforce_fair_resource_policy()
    pipeline = make_pipeline(model_code, params, scaler_name)
    cv_out = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=CV,
        scoring="accuracy",
        n_jobs=N_JOBS_CV,
        return_train_score=False,
    )
    return float(np.mean(cv_out["test_score"]))


def run_tpe_for_model(model_code: str, scaler_name: str) -> Tuple[Dict[str, Any], pd.DataFrame]:
    trial_rows = []
    best_so_far = -np.inf

    def objective(trial: optuna.trial.Trial) -> float:
        start_time = time.perf_counter()
        params = sample_params_tpe(trial, model_code)
        score = objective_from_params(model_code, scaler_name, params)
        runtime_sec = float(time.perf_counter() - start_time)
        trial.set_user_attr("runtime_sec", runtime_sec)
        return score

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def callback(study_obj: optuna.Study, frozen_trial: optuna.trial.FrozenTrial) -> None:
        nonlocal best_so_far
        if frozen_trial.value is None:
            return
        best_so_far = max(best_so_far, float(frozen_trial.value))
        trial_rows.append(
            {
                "Model": model_code,
                "Trial": len(trial_rows) + 1,
                "CV_Mean": float(frozen_trial.value),
                "Best_Score_So_Far": best_so_far,
                "Trial_Runtime_Sec": float(frozen_trial.user_attrs.get("runtime_sec", np.nan)),
            }
        )

    study.optimize(objective, n_trials=N_TRIALS_PER_MODEL, callbacks=[callback], show_progress_bar=False)
    return {
        "Model": model_code,
        "Optimal_Scaler": scaler_name,
        "Best_CV_Mean": float(study.best_value),
        "Best_Trial": int(study.best_trial.number),
        **study.best_trial.params,
    }, pd.DataFrame(trial_rows)


def run_gwo_for_model(model_code: str, scaler_name: str) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rng = np.random.default_rng(RANDOM_STATE)
    dim = len(SEARCH_SPACES[model_code])
    wolves = rng.random((GWO_POPULATION_SIZE, dim))

    alpha_pos   = np.zeros(dim)
    beta_pos    = np.zeros(dim)
    delta_pos   = np.zeros(dim)
    alpha_score = -np.inf
    beta_score  = -np.inf
    delta_score = -np.inf
    alpha_params: Dict[str, Any] = {}
    history_rows = []
    evaluation_counter = 0
    best_iteration = 0
    total_evals = GWO_POPULATION_SIZE * (GWO_ITERATIONS + 1)  # 10 init + 4*10 = 50

    print(f"       GWO progress for {model_code}: 0/{total_evals} evaluations", flush=True)

    # Base implementation kept here so shared helpers remain runnable before later override cells.
    print(f"       GWO {model_code}: initializing population ({GWO_POPULATION_SIZE} wolves)...", flush=True)
    for wolf_idx in range(GWO_POPULATION_SIZE):
        position = wolves[wolf_idx].copy()
        params = decode_gwo_position(position, model_code)
        start_time = time.perf_counter()
        score = objective_from_params(model_code, scaler_name, params)
        runtime_sec = float(time.perf_counter() - start_time)
        evaluation_counter += 1

        if score > alpha_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = alpha_score, alpha_pos.copy()
            alpha_score, alpha_pos = score, position.copy()
            alpha_params = dict(params)
            best_iteration = evaluation_counter
        elif score > beta_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = score, position.copy()
        elif score > delta_score:
            delta_score, delta_pos = score, position.copy()

        history_rows.append(
            {
                "Model": model_code,
                "Iteration": evaluation_counter,
                "CV_Mean": float(score),
                "Best_Score_So_Far": float(alpha_score),
                "Iteration_Runtime_Sec": runtime_sec,
            }
        )

    for outer_iter in range(GWO_ITERATIONS):
        a_value = 2.0 - 2.0 * (outer_iter / max(GWO_ITERATIONS - 1, 1))
        scores = np.full(GWO_POPULATION_SIZE, -np.inf)
        params_list = []

        for wolf_idx in range(GWO_POPULATION_SIZE):
            position = wolves[wolf_idx].copy()
            params = decode_gwo_position(position, model_code)
            start_time = time.perf_counter()
            score = objective_from_params(model_code, scaler_name, params)
            runtime_sec = float(time.perf_counter() - start_time)
            scores[wolf_idx] = score
            params_list.append(params)
            evaluation_counter += 1

            history_rows.append(
                {
                    "Model": model_code,
                    "Iteration": evaluation_counter,
                    "CV_Mean": float(score),
                    "Best_Score_So_Far": float(alpha_score),
                    "Iteration_Runtime_Sec": runtime_sec,
                }
            )

        for wolf_idx in range(GWO_POPULATION_SIZE):
            score = scores[wolf_idx]
            position = wolves[wolf_idx].copy()
            params = params_list[wolf_idx]
            if score > alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = score, position.copy()
                alpha_params = dict(params)
                best_iteration = evaluation_counter - (GWO_POPULATION_SIZE - wolf_idx - 1)
            elif score > beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = score, position.copy()
            elif score > delta_score:
                delta_score, delta_pos = score, position.copy()

        for i in range(GWO_POPULATION_SIZE):
            history_rows[-(GWO_POPULATION_SIZE - i)]["Best_Score_So_Far"] = float(alpha_score)

        new_wolves = np.zeros_like(wolves)
        for wolf_idx in range(GWO_POPULATION_SIZE):
            position = wolves[wolf_idx].copy()

            r1 = rng.random(dim)
            r2 = rng.random(dim)
            A1 = 2.0 * a_value * r1 - a_value
            C1 = 2.0 * r2
            D_alpha = np.abs(C1 * alpha_pos - position)
            X1 = alpha_pos - A1 * D_alpha

            r1 = rng.random(dim)
            r2 = rng.random(dim)
            A2 = 2.0 * a_value * r1 - a_value
            C2 = 2.0 * r2
            D_beta = np.abs(C2 * beta_pos - position)
            X2 = beta_pos - A2 * D_beta

            r1 = rng.random(dim)
            r2 = rng.random(dim)
            A3 = 2.0 * a_value * r1 - a_value
            C3 = 2.0 * r2
            D_delta = np.abs(C3 * delta_pos - position)
            X3 = delta_pos - A3 * D_delta

            new_wolves[wolf_idx] = np.clip((X1 + X2 + X3) / 3.0, 0.0, 1.0)

        wolves = new_wolves

    return {
        "Model": model_code,
        "Optimal_Scaler": scaler_name,
        "Best_CV_Mean": float(alpha_score),
        "Best_Iteration": int(best_iteration),
        **alpha_params,
    }, pd.DataFrame(history_rows)


def select_scaler_for_optimization(model_code: str) -> str:
    scaler_row = baseline_best_scalers_df.loc[baseline_best_scalers_df["Model"] == model_code]
    if scaler_row.empty:
        raise KeyError(f"No baseline scaler found for {model_code}")
    return scaler_row.iloc[0]["Optimal_Scaler"]


METHOD_CONFIG = {
    "TPE": {
        "runner": run_tpe_for_model,
        "history_step_col": "Trial",
        "best_step_col": "Best_Trial",
        "search_label": "trials",
        "step_count": N_TRIALS_PER_MODEL,
    },
    "GWO": {
        "runner": run_gwo_for_model,
        "history_step_col": "Iteration",
        "best_step_col": "Best_Iteration",
        "search_label": "evaluations",
        "step_count": GWO_POPULATION_SIZE * (GWO_ITERATIONS + 1),
    },
}


def run_single_model_optimization(method_name: str, model_code: str, progress_log: Dict[str, Dict[str, str]]) -> Dict[str, Any]:
    config = METHOD_CONFIG[method_name]
    status_key = method_name.lower()
    scaler_name = select_scaler_for_optimization(model_code)
    memory_before = refresh_memory_snapshot()

    print(f"\n{model_code:10s} | Scaler: {scaler_name:10s} | Available RAM: {memory_before['available_gb']:.2f} GB | Kernel RSS: {memory_before['process_mb']:.1f} MB")

    if progress_log.get(model_code, {}).get(status_key) == "completed":
        print(f"       Skipping: {method_name} already completed")
        return {"status": "skipped"}

    enforce_fair_resource_policy()
    collect_runtime_garbage()
    model_start = time.perf_counter()

    try:
        print(f"       Running {config['step_count']} {config['search_label']} with n_jobs_cv={N_JOBS_CV}, n_jobs_opt={N_JOBS_OPT}")
        with ResourceMonitor(f"{method_name} {model_code}"):
            search_start = time.perf_counter()
            best_row, history_df = config["runner"](model_code, scaler_name)
            search_time = time.perf_counter() - search_start

        with ResourceMonitor(f"{method_name} Eval {model_code}"):
            eval_start = time.perf_counter()
            _, summary_df, _ = evaluate_on_split(
                method_name=method_name,
                model_code=model_code,
                scaler_name=scaler_name,
                params={key: value for key, value in best_row.items() if key not in {"Model", "Optimal_Scaler", "Best_CV_Mean", config['best_step_col']}},
                X_train_input=X_train,
                y_train_input=y_train,
                X_test_input=X_test,
                y_test_input=y_test,
            )
            eval_time = time.perf_counter() - eval_start

        total_time = time.perf_counter() - model_start
        timing_row = {
            "Model": model_code,
            f"{method_name}_Search_Sec": float(search_time),
            f"{method_name}_Evaluation_Sec": float(eval_time),
            f"{method_name}_Total_Sec": float(total_time),
            "Best_CV_Mean": float(best_row["Best_CV_Mean"]),
        }

        progress_log.setdefault(model_code, {})[status_key] = "completed"
        save_progress_log(progress_log)
        save_model_checkpoint(
            method_name,
            model_code,
            {
                "best": best_row,
                "timing": timing_row,
                "memory_before": memory_before,
                "memory_after": refresh_memory_snapshot(),
            },
        )

        collect_runtime_garbage()
        return {
            "status": "completed",
            "best": pd.DataFrame([best_row]),
            "history": history_df,
            "summary": summary_df,
            "timing": pd.DataFrame([timing_row]),
        }
    except Exception as exc:
        progress_log.setdefault(model_code, {})[status_key] = f"failed: {str(exc)[:200]}"
        save_progress_log(progress_log)
        collect_runtime_garbage()
        print(f"       FAILED: {str(exc)[:120]}")
        return {"status": "failed", "error": str(exc)}


def run_method_sequentially(method_name: str, model_order: List[str] = MODEL_ORDER) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    config = METHOD_CONFIG[method_name]
    progress_log = load_progress_log()
    best_df, history_df, results_df, timing_df = load_existing_method_outputs(method_name)
    completed = 0
    failed = 0

    print("\n" + "=" * 80)
    print(f"{method_name} optimization (sequential, publication mode)")
    print(f"Fixed resources: n_jobs_cv={FIXED_N_JOBS_CV}, n_jobs_opt={FIXED_N_JOBS_OPT}")
    print("=" * 80)

    eligible_models = [model_code for model_code in model_order if model_code in SEARCH_SPACES]
    method_start = time.perf_counter()

    for index, model_code in enumerate(eligible_models, start=1):
        print(f"\n[{index:2d}/{len(eligible_models)}] {method_name} :: {model_code}")
        result = run_single_model_optimization(method_name, model_code, progress_log)

        if result["status"] == "completed":
            best_df = deduplicate_rows(pd.concat([best_df, result["best"]], ignore_index=True), ["Model"])
            history_df = deduplicate_rows(pd.concat([history_df, result["history"]], ignore_index=True), ["Model", config["history_step_col"]])
            results_df = deduplicate_rows(pd.concat([results_df, result["summary"]], ignore_index=True), ["Method", "Model", "Split"])
            timing_df = deduplicate_rows(pd.concat([timing_df, result["timing"]], ignore_index=True), ["Model"])
            export_method_outputs(method_name, best_df, history_df, results_df, timing_df, print_status=True)
            completed += 1
        elif result["status"] == "failed":
            failed += 1

    elapsed = time.perf_counter() - method_start
    print("\n" + "=" * 80)
    print(f"{method_name} summary: {completed} completed, {failed} failed, elapsed {elapsed:7.2f}s ({elapsed/60:.2f} min)")
    print("=" * 80)

    if not timing_df.empty:
        display(timing_df.sort_values(f"{method_name}_Total_Sec", ascending=False))

    return best_df, history_df, results_df, timing_df


print("Optimization helpers ready: sequential model-by-model workflow with fixed resources.")

Optimization helpers ready: sequential model-by-model workflow with fixed resources.


In [34]:
# =====================================================================
# CELL 8: TPE Optimization - Explicit Single-Model Runner
# =====================================================================

if "tpe_best_df" not in globals():
    tpe_best_df, tpe_history_df, tpe_results_df, tpe_timing_df = load_existing_method_outputs("TPE")


def run_tpe_model(model_code: str) -> Dict[str, Any]:
    """Run TPE for exactly one model and persist outputs incrementally."""
    global tpe_best_df, tpe_history_df, tpe_results_df, tpe_timing_df
    config = METHOD_CONFIG["TPE"]
    progress_log = load_progress_log()

    result = run_single_model_optimization("TPE", model_code, progress_log)

    if result["status"] == "completed":
        tpe_best_df = deduplicate_rows(pd.concat([tpe_best_df, result["best"]], ignore_index=True), ["Model"])
        tpe_history_df = deduplicate_rows(
            pd.concat([tpe_history_df, result["history"]], ignore_index=True),
            ["Model", config["history_step_col"]],
        )
        tpe_results_df = deduplicate_rows(
            pd.concat([tpe_results_df, result["summary"]], ignore_index=True),
            ["Method", "Model", "Split"],
        )
        tpe_timing_df = deduplicate_rows(pd.concat([tpe_timing_df, result["timing"]], ignore_index=True), ["Model"])
        export_method_outputs("TPE", tpe_best_df, tpe_history_df, tpe_results_df, tpe_timing_df, print_status=True)

    return result


print("TPE single-model runner ready.")
print("Call run_tpe_model('<MODEL_CODE>') one model at a time.")
print(f"Allowed models: {', '.join([m for m in MODEL_ORDER if m in SEARCH_SPACES])}")

TPE single-model runner ready.
Call run_tpe_model('<MODEL_CODE>') one model at a time.
Allowed models: LR, LDA, QDA, NB, KNN, LinearSVC, SVM, AdaBoost, RF, GB, XGB, LGBM, CB, SGD


In [35]:
# =====================================================================
# CELL 8B: GWO Optimization - Explicit Single-Model Runner
# =====================================================================

if "gwo_best_df" not in globals():
    gwo_best_df, gwo_history_df, gwo_results_df, gwo_timing_df = load_existing_method_outputs("GWO")


def run_gwo_model(model_code: str) -> Dict[str, Any]:
    """Run GWO for exactly one model and persist outputs incrementally."""
    global gwo_best_df, gwo_history_df, gwo_results_df, gwo_timing_df
    config = METHOD_CONFIG["GWO"]
    progress_log = load_progress_log()

    result = run_single_model_optimization("GWO", model_code, progress_log)

    if result["status"] == "completed":
        gwo_best_df = deduplicate_rows(pd.concat([gwo_best_df, result["best"]], ignore_index=True), ["Model"])
        gwo_history_df = deduplicate_rows(
            pd.concat([gwo_history_df, result["history"]], ignore_index=True),
            ["Model", config["history_step_col"]],
        )
        gwo_results_df = deduplicate_rows(
            pd.concat([gwo_results_df, result["summary"]], ignore_index=True),
            ["Method", "Model", "Split"],
        )
        gwo_timing_df = deduplicate_rows(pd.concat([gwo_timing_df, result["timing"]], ignore_index=True), ["Model"])
        export_method_outputs("GWO", gwo_best_df, gwo_history_df, gwo_results_df, gwo_timing_df, print_status=True)

    return result


print("GWO single-model runner ready.")
print("Call run_gwo_model('<MODEL_CODE>') one model at a time.")
print(f"Allowed models: {', '.join([m for m in MODEL_ORDER if m in SEARCH_SPACES])}")

GWO single-model runner ready.
Call run_gwo_model('<MODEL_CODE>') one model at a time.
Allowed models: LR, LDA, QDA, NB, KNN, LinearSVC, SVM, AdaBoost, RF, GB, XGB, LGBM, CB, SGD


In [68]:
# =====================================================================
# CELL 8C: Live Progress Enhancements (TPE/GWO)
# =====================================================================

TPE_PROGRESS_EVERY = 5
GWO_PROGRESS_EVERY = 5


def run_gwo_for_model(model_code: str, scaler_name: str) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rng = np.random.default_rng(RANDOM_STATE)
    dim = len(SEARCH_SPACES[model_code])

    # ── FIX 1: initialize wolves as uniform [0,1] positions (unchanged, correct) ──
    wolves = rng.random((GWO_POPULATION_SIZE, dim))

    alpha_pos = np.zeros(dim)
    beta_pos  = np.zeros(dim)
    delta_pos = np.zeros(dim)
    alpha_score = -np.inf
    beta_score  = -np.inf
    delta_score = -np.inf
    alpha_params: Dict[str, Any] = {}
    history_rows = []
    evaluation_counter = 0
    best_iteration = 0
    total_evals = GWO_POPULATION_SIZE * (GWO_ITERATIONS + 1)  # 10 init + 4*10 = 50

    # Reset shared early-stop state for this model's run.
    _CURRENT_BEST_SCORE[0] = -np.inf
    _EVAL_COUNTER[0] = 0

    print(f"       GWO progress for {model_code}: 0/{total_evals} evaluations", flush=True)

    # ── FIX 2: evaluate the entire initial population BEFORE any position updates ──
    # Paper Section 3.1: "Calculate the fitness of each search agent"
    # This establishes a valid alpha/beta/delta BEFORE the first iteration begins.
    # Current code skips this — wolves[0] in iter 0 updates using alpha=zeros.
    print(f"       GWO {model_code}: initializing population ({GWO_POPULATION_SIZE} wolves)...", flush=True)
    for wolf_idx in range(GWO_POPULATION_SIZE):
        # Expose current best to objective_from_params for early stopping.
        _CURRENT_BEST_SCORE[0] = alpha_score

        position = wolves[wolf_idx].copy()
        params = decode_gwo_position(position, model_code)
        start_time = time.perf_counter()
        score = objective_from_params(model_code, scaler_name, params)
        runtime_sec = float(time.perf_counter() - start_time)
        evaluation_counter += 1

        # Update leaders from initial population
        if score > alpha_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score,  beta_pos  = alpha_score, alpha_pos.copy()
            alpha_score, alpha_pos = score, position.copy()
            alpha_params = dict(params)
            best_iteration = evaluation_counter
        elif score > beta_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score,  beta_pos  = score, position.copy()
        elif score > delta_score:
            delta_score, delta_pos = score, position.copy()

        history_rows.append({
            "Model": model_code,
            "Iteration": evaluation_counter,
            "CV_Mean": float(score),
            "Best_Score_So_Far": float(alpha_score),
            "Iteration_Runtime_Sec": runtime_sec,
        })

        if evaluation_counter == 1 or evaluation_counter % GWO_PROGRESS_EVERY == 0:
            print(
                f"       GWO {model_code}: init {wolf_idx+1}/{GWO_POPULATION_SIZE} | "
                f"last={score:.5f} | best={alpha_score:.5f}",
                flush=True,
            )

    print(
        f"       GWO {model_code}: init done → "
        f"α={alpha_score:.5f} β={beta_score:.5f} δ={delta_score:.5f}",
        flush=True,
    )

    # ── Main GWO loop ──────────────────────────────────────────────────────────────
    for outer_iter in range(GWO_ITERATIONS):

        # ── FIX 3: a decays from 2→0 over iterations (already correct in current code) ──
        # Paper eq. 3.5: a decreases linearly from 2 to 0
        a_value = 2.0 - 2.0 * (outer_iter / max(GWO_ITERATIONS - 1, 1))

        # ── FIX 4: separate evaluation pass and position-update pass ──────────────
        # Pass A: evaluate ALL wolves, collect scores
        scores = np.full(GWO_POPULATION_SIZE, -np.inf)
        runtimes = np.zeros(GWO_POPULATION_SIZE)
        params_list = []
        for wolf_idx in range(GWO_POPULATION_SIZE):
            # Expose current best to objective_from_params for early stopping.
            _CURRENT_BEST_SCORE[0] = alpha_score

            position = wolves[wolf_idx].copy()
            params = decode_gwo_position(position, model_code)
            start_time = time.perf_counter()
            score = objective_from_params(model_code, scaler_name, params)
            runtimes[wolf_idx] = float(time.perf_counter() - start_time)
            scores[wolf_idx] = score
            params_list.append(params)
            evaluation_counter += 1

            history_rows.append({
                "Model": model_code,
                "Iteration": evaluation_counter,
                "CV_Mean": float(score),
                "Best_Score_So_Far": float(alpha_score),   # best known so far (from prev iters)
                "Iteration_Runtime_Sec": runtimes[wolf_idx],
            })

            if evaluation_counter % GWO_PROGRESS_EVERY == 0 or evaluation_counter == total_evals:
                print(
                    f"       GWO {model_code}: {evaluation_counter:2d}/{total_evals} | "
                    f"iter={outer_iter + 1}/{GWO_ITERATIONS} | last={score:.5f} | best={alpha_score:.5f}",
                    flush=True,
                )

        # Pass B: update α, β, δ from ALL scores of this iteration
        for wolf_idx in range(GWO_POPULATION_SIZE):
            score    = scores[wolf_idx]
            position = wolves[wolf_idx].copy()
            params   = params_list[wolf_idx]
            if score > alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score,  beta_pos  = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = score, position.copy()
                alpha_params = dict(params)
                best_iteration = evaluation_counter - (GWO_POPULATION_SIZE - wolf_idx - 1)
            elif score > beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score,  beta_pos  = score, position.copy()
            elif score > delta_score:
                delta_score, delta_pos = score, position.copy()

        # Update Best_Score_So_Far retroactively for this iteration's history rows
        # (rows were written before leader update, so patch them now)
        for i in range(GWO_POPULATION_SIZE):
            history_rows[-(GWO_POPULATION_SIZE - i)]["Best_Score_So_Far"] = float(alpha_score)

        print(
            f"       GWO {model_code}: iter {outer_iter+1}/{GWO_ITERATIONS} done → "
            f"α={alpha_score:.5f} β={beta_score:.5f} δ={delta_score:.5f}",
            flush=True,
        )

        # Pass C: update ALL positions using the SAME α, β, δ
        new_wolves = np.zeros_like(wolves)
        for wolf_idx in range(GWO_POPULATION_SIZE):
            position = wolves[wolf_idx].copy()

            r1 = rng.random(dim);  r2 = rng.random(dim)
            A1 = 2.0 * a_value * r1 - a_value
            C1 = 2.0 * r2
            D_alpha = np.abs(C1 * alpha_pos - position)
            X1 = alpha_pos - A1 * D_alpha

            r1 = rng.random(dim);  r2 = rng.random(dim)
            A2 = 2.0 * a_value * r1 - a_value
            C2 = 2.0 * r2
            D_beta = np.abs(C2 * beta_pos - position)
            X2 = beta_pos - A2 * D_beta

            r1 = rng.random(dim);  r2 = rng.random(dim)
            A3 = 2.0 * a_value * r1 - a_value
            C3 = 2.0 * r2
            D_delta = np.abs(C3 * delta_pos - position)
            X3 = delta_pos - A3 * D_delta

            new_wolves[wolf_idx] = np.clip((X1 + X2 + X3) / 3.0, 0.0, 1.0)

        wolves = new_wolves

    return {
        "Model": model_code,
        "Optimal_Scaler": scaler_name,
        "Best_CV_Mean": float(alpha_score),
        "Best_Iteration": int(best_iteration),
        **alpha_params,
    }, pd.DataFrame(history_rows)


# Ensure single-model runners use the enhanced progress versions.
METHOD_CONFIG["TPE"]["runner"] = run_tpe_for_model
METHOD_CONFIG["GWO"]["runner"] = run_gwo_for_model

print("Live progress enhancements enabled.")
print(f"TPE update frequency: every {TPE_PROGRESS_EVERY} trials")
print(f"GWO update frequency: every {GWO_PROGRESS_EVERY} evaluations")

Live progress enhancements enabled.
TPE update frequency: every 5 trials
GWO update frequency: every 5 evaluations


In [78]:
# =====================================================================
# CELL 8C2: Free Memory Before Each Model Run
# =====================================================================


def collect_runtime_garbage() -> None:
    """Aggressive memory cleanup with visibility before/after optimization runs."""
    collected = gc.collect()
    vm = psutil.virtual_memory()
    process_mb = psutil.Process(os.getpid()).memory_info().rss / (1024**2)

    print(
        f"       Memory cleanup -> gc_collected={collected}, "
        f"available={vm.available / (1024**3):.2f} GB, "
        f"used={vm.percent:.1f}%, rss={process_mb:.1f} MB",
        flush=True,
    )


print("Free-memory hook enabled: cleanup runs automatically before each model optimization.")

Free-memory hook enabled: cleanup runs automatically before each model optimization.


In [69]:
# =====================================================================
# CELL 8C3: Full-Core Mode + Fold-Level Live Progress + Early Stopping
# =====================================================================

FORCE_FULL_CORE_MODE = True
SHOW_FOLD_PROGRESS = True
TPE_PROGRESS_EVERY = 1
GWO_PROGRESS_EVERY = 1

# ── Early stopping: abandon remaining folds on clearly losing wolves ──────────
# Applied after EARLY_STOP_MIN_FOLD folds; only kicks in once a valid best exists.
EARLY_STOP_MIN_FOLD = 2      # earliest fold index at which we can stop
EARLY_STOP_GAP      = 0.045  # stop if running_mean is >4.5% below current best

# Shared state: GWO/TPE loop writes current best here before each wolf call.
# objective_from_params reads it for early stopping without changing its signature.
_CURRENT_BEST_SCORE: List[float] = [-np.inf]

# Per-run evaluation counter for periodic RAM re-poll (mutable list = shared ref).
_EVAL_COUNTER: List[int] = [0]

if FORCE_FULL_CORE_MODE:
    # Threads per model fit are controlled via OMP/MKL env vars (N_JOBS_OPT).
    # The manual fold loop is inherently serial, so N_JOBS_CV stays 1.
    N_JOBS_CV = 1
    N_JOBS_OPT = max(1, CPU_COUNT)
    FIXED_N_JOBS_CV = N_JOBS_CV
    FIXED_N_JOBS_OPT = N_JOBS_OPT


def _repoll_resources(model_code: str) -> None:
    """Silently refresh thread-count env vars based on current available RAM.
    Called every 10 evaluations to adapt to memory changes mid-run."""
    global N_JOBS_OPT, FIXED_N_JOBS_OPT
    new_jobs = choose_n_jobs_opt(model_code, CPU_COUNT)
    if new_jobs != N_JOBS_OPT:
        N_JOBS_OPT = new_jobs
        FIXED_N_JOBS_OPT = new_jobs
        os.environ["OMP_NUM_THREADS"]     = str(new_jobs)
        os.environ["MKL_NUM_THREADS"]     = str(new_jobs)
        os.environ["OPENBLAS_NUM_THREADS"] = str(new_jobs)
        avail_gb = psutil.virtual_memory().available / (1024 ** 3)
        print(f"          [RAM repoll] free={avail_gb:.2f} GB → n_jobs_opt updated to {new_jobs}", flush=True)


def objective_from_params(model_code: str, scaler_name: str, params: Dict[str, Any]) -> float:
    """Manual CV objective with fold-level progress output and early stopping."""
    _EVAL_COUNTER[0] += 1
    if _EVAL_COUNTER[0] % 10 == 0:
        _repoll_resources(model_code)
    else:
        enforce_fair_resource_policy()

    fold_scores: List[float] = []

    for fold_idx, (fit_idx, val_idx) in enumerate(CV.split(X_train, y_train), start=1):
        fold_start = time.perf_counter()

        pipeline = make_pipeline(model_code, params, scaler_name)
        X_fit = X_train.iloc[fit_idx]
        y_fit = y_train.iloc[fit_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        pipeline.fit(X_fit, y_fit)
        y_pred = pipeline.predict(X_val)
        fold_acc = float(accuracy_score(y_val, y_pred))
        fold_scores.append(fold_acc)

        if SHOW_FOLD_PROGRESS:
            elapsed = time.perf_counter() - fold_start
            print(
                f"          fold {fold_idx}/{N_SPLITS} | "
                f"acc={fold_acc:.5f} | running_mean={np.mean(fold_scores):.5f} | "
                f"time={elapsed:.2f}s",
                flush=True,
            )

        # ── Early stopping ──────────────────────────────────────────────────────
        # Skip remaining folds if this wolf is clearly not competitive.
        # Condition: after EARLY_STOP_MIN_FOLD folds and a valid best exists,
        # abandon if the gap to current best exceeds EARLY_STOP_GAP.
        if fold_idx >= EARLY_STOP_MIN_FOLD and _CURRENT_BEST_SCORE[0] > -np.inf:
            gap = _CURRENT_BEST_SCORE[0] - float(np.mean(fold_scores))
            if gap > EARLY_STOP_GAP:
                if SHOW_FOLD_PROGRESS:
                    print(
                        f"          [EarlyStop] running_mean={np.mean(fold_scores):.5f} | "
                        f"gap={gap:.4f} > {EARLY_STOP_GAP} | best={_CURRENT_BEST_SCORE[0]:.5f} "
                        f"→ skip folds {fold_idx + 1}-{N_SPLITS}",
                        flush=True,
                    )
                break

    return float(np.mean(fold_scores))


print("Full-core mode enabled for optimization.")
print(f"n_jobs_cv={N_JOBS_CV}, n_jobs_opt={N_JOBS_OPT} (all logical cores)")
print("Live fold-level progress enabled.")
print(f"Early stopping: gap > {EARLY_STOP_GAP} after fold {EARLY_STOP_MIN_FOLD}+")
print(f"RAM re-poll: every 10 evaluations")

Full-core mode enabled for optimization.
n_jobs_cv=1, n_jobs_opt=8 (all logical cores)
Live fold-level progress enabled.
Early stopping: gap > 0.045 after fold 2+
RAM re-poll: every 10 evaluations


In [70]:
# ============================================================
# Adaptive resource policy: heavy models run one-at-a-time with
# RAM-friendly thread limits
# ============================================================
import os
import psutil

MODEL_ORDER = ["LR", "LDA", "QDA", "NB", "KNN", "LinearSVC", "SVM", "AdaBoost", "RF", "GB", "XGB", "LGBM", "CB", "SGD"]

HEAVY_MODELS = {"KNN", "SVM", "RF", "GB", "XGB", "LGBM", "CB"}
MEDIUM_MODELS = {"AdaBoost", "LinearSVC", "SGD"}
LIGHT_MODELS = {"LR", "LDA", "QDA", "NB"}

MODEL_THREAD_CAP = {
    "CB": 2,
    "XGB": 2,
    "LGBM": 2,
    "RF": 3,
    "GB": 2,
    "KNN": 2,
    "SVM": 2,
    "AdaBoost": 3,
    "LinearSVC": 3,
    "SGD": 3,
    "LR": 4,
    "LDA": 4,
    "QDA": 4,
    "NB": 4,
}

def _ram_based_cap(avail_gb: float) -> int:
    if avail_gb < 2.5:
        return 1
    if avail_gb < 4.0:
        return 2
    if avail_gb < 5.5:   # was 6.0 — lowered to avoid cliff-edge at ~5.9 GB
        return 3
    return 4

def choose_n_jobs_opt(model_code: str, cpu_count: int) -> int:
    avail_gb = psutil.virtual_memory().available / (1024 ** 3)
    ram_cap = _ram_based_cap(avail_gb)
    cpu_cap = max(1, cpu_count - 1)  # keep one core free for OS/kernel
    model_cap = MODEL_THREAD_CAP.get(model_code, 2)

    if model_code in HEAVY_MODELS:
        return max(1, min(model_cap, ram_cap, cpu_cap))
    if model_code in MEDIUM_MODELS:
        return max(1, min(model_cap, ram_cap + 1, cpu_cap))
    return max(1, min(model_cap, ram_cap + 2, cpu_cap))

def apply_resource_policy_for_model(model_code: str):
    global N_JOBS_CV, N_JOBS_OPT, FIXED_N_JOBS_CV, FIXED_N_JOBS_OPT

    # Keep outer CV serial to avoid nested oversubscription.
    # Thread parallelism is handled via OMP/MKL env vars (N_JOBS_OPT).
    N_JOBS_CV = 1
    N_JOBS_OPT = choose_n_jobs_opt(model_code, CPU_COUNT)

    FIXED_N_JOBS_CV = N_JOBS_CV
    FIXED_N_JOBS_OPT = N_JOBS_OPT

    # Align threaded math backends with chosen n_jobs
    os.environ["OMP_NUM_THREADS"] = str(N_JOBS_OPT)
    os.environ["MKL_NUM_THREADS"] = str(N_JOBS_OPT)
    os.environ["OPENBLAS_NUM_THREADS"] = str(N_JOBS_OPT)

    avail_gb = psutil.virtual_memory().available / (1024 ** 3)
    group = "HEAVY" if model_code in HEAVY_MODELS else ("MEDIUM" if model_code in MEDIUM_MODELS else "LIGHT")
    print(
        f"[ResourcePolicy] model={model_code} ({group}) | free_ram={avail_gb:.2f} GB | "
        f"n_jobs_cv={N_JOBS_CV} | n_jobs_opt={N_JOBS_OPT}",
        flush=True,
    )

# Optional wrappers (use these instead of run_*_model directly)
def run_gwo_model_with_policy(model_code: str):
    apply_resource_policy_for_model(model_code)
    return run_gwo_model(model_code)

def run_tpe_model_with_policy(model_code: str):
    apply_resource_policy_for_model(model_code)
    return run_tpe_model(model_code)

## Cell 8A: Explicit TPE Per-Model Calls

Run each code cell below manually to execute one model at a time.

In [25]:
run_tpe_model("LGBM")


LGBM       | Scaler: Raw        | Available RAM: 4.38 GB | Kernel RSS: 228.2 MB
       Memory cleanup -> gc_collected=67, available=4.37 GB, used=72.2%, rss=228.2 MB
       Running 50 trials with n_jobs_cv=2, n_jobs_opt=2
       TPE progress for LGBM: 0/50 trials
       TPE LGBM:  1/50 | last=0.99148 | best=0.99148
       TPE LGBM:  5/50 | last=0.93225 | best=0.99148
       TPE LGBM: 10/50 | last=0.92915 | best=0.99265
       TPE LGBM: 15/50 | last=0.99375 | best=0.99375
       TPE LGBM: 20/50 | last=0.99517 | best=0.99517
       TPE LGBM: 25/50 | last=0.99656 | best=0.99656
       TPE LGBM: 30/50 | last=0.96560 | best=0.99667
       TPE LGBM: 35/50 | last=0.99769 | best=0.99769
       TPE LGBM: 40/50 | last=0.99712 | best=0.99769
       TPE LGBM: 45/50 | last=0.92529 | best=0.99769
       TPE LGBM: 50/50 | last=0.99406 | best=0.99769
  [TPE LGBM] Time: 2572.02s | Memory Δ:  -142.7MB | Peak:    85.6MB
  [TPE Eval LGBM] Time:  212.04s | Memory Δ:    +1.5MB | Peak:   218.9MB
       Memo

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_estimators  learning_rate  \
 0  LGBM            Raw      0.997687          34           361       0.426536   
 
    max_depth  num_leaves  min_child_samples  subsample  colsample_bytree  \
 0         15          41                 25   0.824208          0.912309   
 
       reg_alpha  reg_lambda  
 0  1.336780e-07    0.000708  ,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0   LGBM      1  0.991479           0.991479        1375.108143
 1   LGBM      2  0.957854           0.991479          51.380237
 2   LGBM      3  0.961104           0.991479          10.438176
 3   LGBM      4  0.955854           0.991479           4.676073
 4   LGBM      5  0.932250           0.991479          10.678451
 5   LGBM      6  0.978021           0.991479          26.943868
 6   LGBM      7  0.992646           0.992646           6.293340
 7   LGBM      8  0.956063           0.992646       

In [14]:
run_tpe_model("LR")


LR         | Scaler: Power      | Available RAM: 2.18 GB | Kernel RSS: 23.0 MB
       Memory cleanup -> gc_collected=76, available=2.03 GB, used=87.1%, rss=143.5 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.87573 | running_mean=0.87573 | time=1.14s
          fold 2/5 | acc=0.87885 | running_mean=0.87729 | time=0.68s
          fold 3/5 | acc=0.87479 | running_mean=0.87646 | time=0.68s
          fold 4/5 | acc=0.87792 | running_mean=0.87682 | time=0.78s
          fold 5/5 | acc=0.87750 | running_mean=0.87696 | time=0.73s
          fold 1/5 | acc=0.89479 | running_mean=0.89479 | time=1.18s
          fold 2/5 | acc=0.89594 | running_mean=0.89536 | time=1.66s
          fold 3/5 | acc=0.89292 | running_mean=0.89455 | time=2.97s
          fold 4/5 | acc=0.89792 | running_mean=0.89539 | time=3.14s
          fold 5/5 | acc=0.90083 | running_mean=0.89648 | time=2.57s
          fold 1/5 | acc=0.78052 | running_mean=0.78052 | time=1.14s
          fold 2/5 

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial         C penalty
 0    LR          Power      0.897312          20  3.703326      l1,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0     LR      1  0.876958           0.876958           4.057603
 1     LR      2  0.896479           0.896479          11.549645
 2     LR      3  0.781979           0.896479           6.856394
 3     LR      4  0.895354           0.896479           7.396691
 4     LR      5  0.897271           0.897271          10.931673
 5     LR      6  0.851333           0.897271           5.551862
 6     LR      7  0.881708           0.897271           4.912265
 7     LR      8  0.846500           0.897271           8.488905
 8     LR      9  0.893771           0.897271           9.344094
 9     LR     10  0.895542           0.897271           9.560997
 10    LR     11  0.897250           0.897271           9.910697
 11    LR     12  0.897250           0.8

In [47]:
run_tpe_model("LDA")


LDA        | Scaler: Power      | Available RAM: 3.56 GB | Kernel RSS: 222.0 MB
       Memory cleanup -> gc_collected=0, available=3.53 GB, used=77.6%, rss=222.0 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.89510 | running_mean=0.89510 | time=0.76s
          fold 2/5 | acc=0.89427 | running_mean=0.89469 | time=0.47s
          fold 3/5 | acc=0.89271 | running_mean=0.89403 | time=0.57s
          fold 4/5 | acc=0.89500 | running_mean=0.89427 | time=0.50s
          fold 5/5 | acc=0.89948 | running_mean=0.89531 | time=0.40s
          fold 1/5 | acc=0.89510 | running_mean=0.89510 | time=0.66s
          fold 2/5 | acc=0.89427 | running_mean=0.89469 | time=0.46s
          fold 3/5 | acc=0.89271 | running_mean=0.89403 | time=0.46s
          fold 4/5 | acc=0.89500 | running_mean=0.89427 | time=0.39s
          fold 5/5 | acc=0.89948 | running_mean=0.89531 | time=0.50s
          fold 1/5 | acc=0.89510 | running_mean=0.89510 | time=0.50s
          fold 2/5 

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial solver
 0   LDA          Power      0.895312           0   lsqr,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0    LDA      1  0.895312           0.895312           2.734593
 1    LDA      2  0.895312           0.895312           2.494715
 2    LDA      3  0.895312           0.895312           2.495437
 3    LDA      4  0.895312           0.895312           2.377044
 4    LDA      5  0.895312           0.895312           2.280556
 5    LDA      6  0.895312           0.895312           2.470868
 6    LDA      7  0.895312           0.895312           2.200756
 7    LDA      8  0.895312           0.895312           2.369751
 8    LDA      9  0.895312           0.895312           2.296578
 9    LDA     10  0.895312           0.895312           2.343263
 10   LDA     11  0.895312           0.895312           2.236792
 11   LDA     12  0.895312           0.895312           2.4611

In [13]:
run_tpe_model("QDA")


QDA        | Scaler: Power      | Available RAM: 2.22 GB | Kernel RSS: 21.1 MB
       Skipping: TPE already completed


{'status': 'skipped'}

In [49]:
run_tpe_model("NB")


NB         | Scaler: Power      | Available RAM: 3.63 GB | Kernel RSS: 222.2 MB
       Memory cleanup -> gc_collected=0, available=3.61 GB, used=77.1%, rss=222.2 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.84448 | running_mean=0.84448 | time=0.54s
          fold 2/5 | acc=0.84521 | running_mean=0.84484 | time=0.42s
          fold 3/5 | acc=0.84292 | running_mean=0.84420 | time=0.60s
          fold 4/5 | acc=0.84292 | running_mean=0.84388 | time=0.46s
          fold 5/5 | acc=0.83969 | running_mean=0.84304 | time=0.43s
          fold 1/5 | acc=0.84448 | running_mean=0.84448 | time=0.48s
          fold 2/5 | acc=0.84521 | running_mean=0.84484 | time=0.39s
          fold 3/5 | acc=0.84292 | running_mean=0.84420 | time=0.41s
          fold 4/5 | acc=0.84292 | running_mean=0.84388 | time=0.47s
          fold 5/5 | acc=0.83969 | running_mean=0.84304 | time=0.32s
          fold 1/5 | acc=0.84448 | running_mean=0.84448 | time=0.35s
          fold 2/5 

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  var_smoothing
 0    NB          Power      0.843042           0   1.767017e-10,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0     NB      1  0.843042           0.843042           2.461328
 1     NB      2  0.843042           0.843042           2.084999
 2     NB      3  0.843042           0.843042           2.154166
 3     NB      4  0.843042           0.843042           2.060119
 4     NB      5  0.843042           0.843042           2.389869
 5     NB      6  0.843042           0.843042           2.373777
 6     NB      7  0.843042           0.843042           2.703418
 7     NB      8  0.843042           0.843042           1.714193
 8     NB      9  0.843042           0.843042           1.435204
 9     NB     10  0.843042           0.843042           1.522638
 10    NB     11  0.843042           0.843042           1.589957
 11    NB     12  0.843042           0.843042 

In [39]:
run_tpe_model("KNN")


KNN        | Scaler: Raw        | Available RAM: 2.72 GB | Kernel RSS: 242.3 MB
       Memory cleanup -> gc_collected=419, available=2.71 GB, used=82.8%, rss=242.3 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.93750 | running_mean=0.93750 | time=2.85s
          fold 2/5 | acc=0.94094 | running_mean=0.93922 | time=2.98s
          fold 3/5 | acc=0.93937 | running_mean=0.93927 | time=5.46s
          fold 4/5 | acc=0.93656 | running_mean=0.93859 | time=6.71s
          fold 5/5 | acc=0.93677 | running_mean=0.93823 | time=4.13s
          fold 1/5 | acc=0.94573 | running_mean=0.94573 | time=9.45s
          fold 2/5 | acc=0.94479 | running_mean=0.94526 | time=8.64s
          fold 3/5 | acc=0.94531 | running_mean=0.94528 | time=7.45s
          fold 4/5 | acc=0.94312 | running_mean=0.94474 | time=7.36s
          fold 5/5 | acc=0.94333 | running_mean=0.94446 | time=7.49s
          fold 1/5 | acc=0.93771 | running_mean=0.93771 | time=2.51s
          fold 2/

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_neighbors   weights     metric  p
 0   KNN            Raw      0.948792          15           14  distance  manhattan  2,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0    KNN      1  0.938229           0.938229          22.141844
 1    KNN      2  0.944458           0.944458          40.416381
 2    KNN      3  0.940771           0.944458          13.994381
 3    KNN      4  0.944792           0.944792          25.011768
 4    KNN      5  0.936625           0.944792          58.935730
 5    KNN      6  0.937708           0.944792          19.460663
 6    KNN      7  0.904250           0.944792           9.520929
 7    KNN      8  0.935417           0.944792          10.341043
 8    KNN      9  0.939042           0.944792          25.084353
 9    KNN     10  0.948000           0.948000          28.097034
 10   KNN     11  0.938229           0.948000          55.948703
 1

In [40]:
run_tpe_model("LinearSVC")


LinearSVC  | Scaler: Power      | Available RAM: 2.19 GB | Kernel RSS: 237.5 MB
       Memory cleanup -> gc_collected=0, available=2.19 GB, used=86.1%, rss=237.5 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.88156 | running_mean=0.88156 | time=0.43s
          fold 2/5 | acc=0.88562 | running_mean=0.88359 | time=0.52s
          fold 3/5 | acc=0.88094 | running_mean=0.88271 | time=0.48s
          fold 4/5 | acc=0.88313 | running_mean=0.88281 | time=0.56s
          fold 5/5 | acc=0.88177 | running_mean=0.88260 | time=0.55s
          fold 1/5 | acc=0.89219 | running_mean=0.89219 | time=0.55s
          fold 2/5 | acc=0.89635 | running_mean=0.89427 | time=0.51s
          fold 3/5 | acc=0.89042 | running_mean=0.89299 | time=0.53s
          fold 4/5 | acc=0.89531 | running_mean=0.89357 | time=0.54s
          fold 5/5 | acc=0.89927 | running_mean=0.89471 | time=0.56s
          fold 1/5 | acc=0.84375 | running_mean=0.84375 | time=0.49s
          fold 2/5 

{'status': 'completed',
 'best':        Model Optimal_Scaler  Best_CV_Mean  Best_Trial          C   loss
 0  LinearSVC          Power      0.897312          35  31.523392  hinge,
 'history':         Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0   LinearSVC      1  0.882604           0.882604           2.554715
 1   LinearSVC      2  0.894708           0.894708           2.705660
 2   LinearSVC      3  0.844854           0.894708           2.408250
 3   LinearSVC      4  0.897042           0.897042           3.438370
 4   LinearSVC      5  0.896958           0.897042           9.717074
 5   LinearSVC      6  0.876104           0.897042           3.306910
 6   LinearSVC      7  0.893958           0.897042           3.048569
 7   LinearSVC      8  0.871813           0.897042           2.832639
 8   LinearSVC      9  0.890104           0.897042           2.952895
 9   LinearSVC     10  0.892458           0.897042           2.740367
 10  LinearSVC     11  0.897021         

In [46]:
run_tpe_model("SVM")


SVM        | Scaler: Standard   | Available RAM: 6.03 GB | Kernel RSS: 217.7 MB
       Memory cleanup -> gc_collected=67, available=6.03 GB, used=61.7%, rss=217.7 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.97969 | running_mean=0.97969 | time=46.62s
          fold 2/5 | acc=0.97708 | running_mean=0.97839 | time=44.79s
          fold 3/5 | acc=0.97573 | running_mean=0.97750 | time=44.58s
          fold 4/5 | acc=0.97844 | running_mean=0.97773 | time=45.44s
          fold 5/5 | acc=0.97917 | running_mean=0.97802 | time=45.17s
          fold 1/5 | acc=0.98427 | running_mean=0.98427 | time=296.68s
          fold 2/5 | acc=0.98375 | running_mean=0.98401 | time=344.14s
          fold 3/5 | acc=0.98104 | running_mean=0.98302 | time=285.99s
          fold 4/5 | acc=0.98104 | running_mean=0.98253 | time=297.49s
          fold 5/5 | acc=0.98281 | running_mean=0.98258 | time=337.51s
          fold 1/5 | acc=0.96281 | running_mean=0.96281 | time=91.78s
  

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial kernel           C gamma  degree     coef0
 0   SVM       Standard      0.994354           5   poly  285.708008  auto       3  0.520068,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0    SVM      1  0.978021           0.978021         226.624020
 1    SVM      2  0.982583           0.982583        1561.825985
 2    SVM      3  0.963021           0.982583         456.067668
 3    SVM      4  0.979688           0.982583         303.766006
 4    SVM      5  0.739354           0.982583         668.378727
 5    SVM      6  0.994354           0.994354         573.958113
 6    SVM      7  0.719688           0.994354         880.973791
 7    SVM      8  0.966437           0.994354         416.505895
 8    SVM      9  0.987583           0.994354         370.393064
 9    SVM     10  0.987396           0.994354         250.399447
 10   SVM     11  0.810229           0.994354        18

In [50]:
run_tpe_model("AdaBoost")


AdaBoost   | Scaler: Raw        | Available RAM: 3.51 GB | Kernel RSS: 222.5 MB
       Memory cleanup -> gc_collected=0, available=3.51 GB, used=77.7%, rss=222.5 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.93802 | running_mean=0.93802 | time=16.32s
          fold 2/5 | acc=0.93854 | running_mean=0.93828 | time=13.30s
          fold 3/5 | acc=0.94104 | running_mean=0.93920 | time=13.18s
          fold 4/5 | acc=0.93854 | running_mean=0.93904 | time=13.13s
          fold 5/5 | acc=0.93896 | running_mean=0.93902 | time=14.09s
          fold 1/5 | acc=0.83281 | running_mean=0.83281 | time=7.82s
          fold 2/5 | acc=0.84021 | running_mean=0.83651 | time=7.34s
          fold 3/5 | acc=0.83698 | running_mean=0.83667 | time=7.35s
          fold 4/5 | acc=0.83760 | running_mean=0.83690 | time=9.75s
          fold 5/5 | acc=0.83177 | running_mean=0.83588 | time=10.05s
          fold 1/5 | acc=0.91240 | running_mean=0.91240 | time=21.72s
          fo

{'status': 'completed',
 'best':       Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_estimators  \
 0  AdaBoost            Raw      0.950979          45           401   
 
    learning_rate algorithm  
 0       1.809305   SAMME.R  ,
 'history':        Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0   AdaBoost      1  0.939021           0.939021          70.041766
 1   AdaBoost      2  0.835875           0.939021          42.316324
 2   AdaBoost      3  0.912813           0.939021         111.654005
 3   AdaBoost      4  0.863375           0.939021         187.595616
 4   AdaBoost      5  0.872854           0.939021          86.195142
 5   AdaBoost      6  0.849688           0.939021         153.276545
 6   AdaBoost      7  0.914104           0.939021         110.832179
 7   AdaBoost      8  0.842938           0.939021         123.464680
 8   AdaBoost      9  0.919542           0.939021          29.148242
 9   AdaBoost     10  0.838479           0.939021          66.

In [41]:
run_tpe_model("RF")


RF         | Scaler: MinMax     | Available RAM: 2.11 GB | Kernel RSS: 203.9 MB
       Memory cleanup -> gc_collected=0, available=2.12 GB, used=86.5%, rss=203.9 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.94052 | running_mean=0.94052 | time=10.58s
          fold 2/5 | acc=0.94490 | running_mean=0.94271 | time=9.53s
          fold 3/5 | acc=0.94510 | running_mean=0.94351 | time=8.76s
          fold 4/5 | acc=0.94125 | running_mean=0.94294 | time=9.87s
          fold 5/5 | acc=0.94146 | running_mean=0.94265 | time=9.08s
          fold 1/5 | acc=0.95021 | running_mean=0.95021 | time=22.02s
          fold 2/5 | acc=0.95344 | running_mean=0.95182 | time=25.14s
          fold 3/5 | acc=0.95260 | running_mean=0.95208 | time=28.98s
          fold 4/5 | acc=0.94979 | running_mean=0.95151 | time=27.77s
          fold 5/5 | acc=0.94948 | running_mean=0.95110 | time=24.19s
          fold 1/5 | acc=0.90146 | running_mean=0.90146 | time=18.09s
          fo

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_estimators  max_depth  min_samples_split  min_samples_leaf max_features
 0    RF         MinMax      0.970604          46           233         18                  2                 2         None,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0     RF      1  0.942646           0.942646          47.843787
 1     RF      2  0.951104           0.951104         128.170631
 2     RF      3  0.904292           0.951104          86.693251
 3     RF      4  0.931438           0.951104          28.158073
 4     RF      5  0.824563           0.951104         119.131775
 5     RF      6  0.942458           0.951104         346.789984
 6     RF      7  0.955688           0.955688          71.780327
 7     RF      8  0.950771           0.955688         133.568251
 8     RF      9  0.904708           0.955688          56.279376
 9     RF     10  0.934958           0.955688         19

In [42]:
run_tpe_model("GB")


GB         | Scaler: Raw        | Available RAM: 4.75 GB | Kernel RSS: 189.4 MB
       Memory cleanup -> gc_collected=0, available=4.71 GB, used=70.1%, rss=189.4 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.99083 | running_mean=0.99083 | time=135.65s
          fold 2/5 | acc=0.98927 | running_mean=0.99005 | time=215.46s
          fold 3/5 | acc=0.99125 | running_mean=0.99045 | time=184.83s
          fold 4/5 | acc=0.98938 | running_mean=0.99018 | time=159.06s
          fold 5/5 | acc=0.99062 | running_mean=0.99027 | time=198.35s
          fold 1/5 | acc=0.98313 | running_mean=0.98313 | time=56.74s
          fold 2/5 | acc=0.98604 | running_mean=0.98458 | time=67.59s
          fold 3/5 | acc=0.98625 | running_mean=0.98514 | time=71.88s
          fold 4/5 | acc=0.98719 | running_mean=0.98565 | time=106.99s
          fold 5/5 | acc=0.98531 | running_mean=0.98558 | time=85.17s
          fold 1/5 | acc=0.96240 | running_mean=0.96240 | time=255.49s
 

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_estimators  learning_rate  max_depth  min_samples_split  min_samples_leaf  subsample
 0    GB            Raw      0.995813          34           447        0.40012          5                  4                 8   0.728487,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0     GB      1  0.990271           0.990271         893.379563
 1     GB      2  0.985583           0.990271         388.390858
 2     GB      3  0.961875           0.990271        1217.823000
 3     GB      4  0.980708           0.990271        1233.276206
 4     GB      5  0.989542           0.990271         461.036835
 5     GB      6  0.902313           0.990271         499.855642
 6     GB      7  0.972417           0.990271         942.538571
 7     GB      8  0.979229           0.990271         172.412765
 8     GB      9  0.985146           0.990271        2225.706093
 9     GB     10  0.992000    

In [43]:
run_tpe_model("XGB")


XGB        | Scaler: MinMax     | Available RAM: 6.32 GB | Kernel RSS: 196.9 MB
       Memory cleanup -> gc_collected=64, available=6.31 GB, used=59.9%, rss=197.4 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.98594 | running_mean=0.98594 | time=1.62s
          fold 2/5 | acc=0.98667 | running_mean=0.98630 | time=1.82s
          fold 3/5 | acc=0.98646 | running_mean=0.98635 | time=4.27s
          fold 4/5 | acc=0.98719 | running_mean=0.98656 | time=5.43s
          fold 5/5 | acc=0.98656 | running_mean=0.98656 | time=4.24s
          fold 1/5 | acc=0.95531 | running_mean=0.95531 | time=10.42s
          fold 2/5 | acc=0.95813 | running_mean=0.95672 | time=9.17s
          fold 3/5 | acc=0.95917 | running_mean=0.95753 | time=5.82s
          fold 4/5 | acc=0.95490 | running_mean=0.95688 | time=6.30s
          fold 5/5 | acc=0.95583 | running_mean=0.95667 | time=6.21s
          fold 1/5 | acc=0.97042 | running_mean=0.97042 | time=3.35s
          fold 2/

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  n_estimators  learning_rate  max_depth  min_child_weight  subsample  colsample_bytree     gamma  reg_alpha  reg_lambda
 0   XGB         MinMax      0.993563          32           363       0.397021          5                 2   0.639592          0.850623  0.080977    0.00072    0.000004,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0    XGB      1  0.986563           0.986563          17.417580
 1    XGB      2  0.956667           0.986563          37.945979
 2    XGB      3  0.972938           0.986563          24.485854
 3    XGB      4  0.953333           0.986563           9.013771
 4    XGB      5  0.947521           0.986563          22.938708
 5    XGB      6  0.969604           0.986563          21.394929
 6    XGB      7  0.986583           0.986583          15.892393
 7    XGB      8  0.964375           0.986583          16.845733
 8    XGB      9  0.976021     

In [44]:
run_tpe_model("CB")


CB         | Scaler: MinMax     | Available RAM: 4.74 GB | Kernel RSS: 219.8 MB
       Memory cleanup -> gc_collected=119, available=4.73 GB, used=69.9%, rss=219.8 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.98260 | running_mean=0.98260 | time=5.76s
          fold 2/5 | acc=0.98479 | running_mean=0.98370 | time=5.92s
          fold 3/5 | acc=0.98375 | running_mean=0.98372 | time=4.64s
          fold 4/5 | acc=0.98500 | running_mean=0.98404 | time=4.94s
          fold 5/5 | acc=0.98646 | running_mean=0.98452 | time=4.63s
          fold 1/5 | acc=0.96573 | running_mean=0.96573 | time=0.72s
          fold 2/5 | acc=0.97198 | running_mean=0.96885 | time=0.66s
          fold 3/5 | acc=0.97167 | running_mean=0.96979 | time=0.78s
          fold 4/5 | acc=0.97073 | running_mean=0.97003 | time=0.76s
          fold 5/5 | acc=0.96958 | running_mean=0.96994 | time=0.74s
          fold 1/5 | acc=0.92635 | running_mean=0.92635 | time=5.79s
          fold 2/

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial  iterations  learning_rate  depth  l2_leaf_reg  border_count  bagging_temperature
 0    CB         MinMax      0.994021          47         441        0.39283      6     5.920924            51             6.306163,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0     CB      1  0.984521           0.984521          25.919743
 1     CB      2  0.969938           0.984521           3.683115
 2     CB      3  0.929604           0.984521          28.732418
 3     CB      4  0.953792           0.984521          25.887149
 4     CB      5  0.976438           0.984521          17.464252
 5     CB      6  0.895813           0.984521          21.211095
 6     CB      7  0.906479           0.984521          11.213913
 7     CB      8  0.958646           0.984521           2.761256
 8     CB      9  0.964000           0.984521         116.973034
 9     CB     10  0.975979           0.984

In [45]:
run_tpe_model("SGD")


SGD        | Scaler: Power      | Available RAM: 6.17 GB | Kernel RSS: 219.8 MB
       Memory cleanup -> gc_collected=119, available=6.13 GB, used=61.0%, rss=219.8 MB
       Running 50 trials with n_jobs_cv=1, n_jobs_opt=8
          fold 1/5 | acc=0.84458 | running_mean=0.84458 | time=1.06s
          fold 2/5 | acc=0.88562 | running_mean=0.86510 | time=1.34s
          fold 3/5 | acc=0.87729 | running_mean=0.86917 | time=1.02s
          fold 4/5 | acc=0.88948 | running_mean=0.87424 | time=1.01s
          fold 5/5 | acc=0.86625 | running_mean=0.87265 | time=1.14s
          fold 1/5 | acc=0.85521 | running_mean=0.85521 | time=0.51s
          fold 2/5 | acc=0.86083 | running_mean=0.85802 | time=0.50s
          fold 3/5 | acc=0.85635 | running_mean=0.85747 | time=0.51s
          fold 4/5 | acc=0.85802 | running_mean=0.85760 | time=0.53s
          fold 5/5 | acc=0.85531 | running_mean=0.85715 | time=0.54s
          fold 1/5 | acc=0.88646 | running_mean=0.88646 | time=0.71s
          fold 2/

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Trial      loss penalty     alpha  l1_ratio
 0   SGD          Power      0.896104          47  log_loss      l1  0.000208  0.335745,
 'history':    Model  Trial   CV_Mean  Best_Score_So_Far  Trial_Runtime_Sec
 0    SGD      1  0.872646           0.872646           5.584355
 1    SGD      2  0.857146           0.872646           2.607894
 2    SGD      3  0.887042           0.887042           3.667438
 3    SGD      4  0.874458           0.887042           5.235700
 4    SGD      5  0.831958           0.887042           2.946652
 5    SGD      6  0.886854           0.887042           4.105096
 6    SGD      7  0.891979           0.891979           2.842161
 7    SGD      8  0.892896           0.892896           2.821786
 8    SGD      9  0.890438           0.892896           4.611792
 9    SGD     10  0.889458           0.892896           5.399378
 10   SGD     11  0.892917           0.892917           2.800921


## Cell 8D: Explicit GWO Per-Model Calls

Run each code cell below manually to execute one model at a time.

In [45]:
run_gwo_model("LGBM")


LGBM       | Scaler: Raw        | Available RAM: 5.53 GB | Kernel RSS: 26.1 MB
       Memory cleanup -> gc_collected=20, available=5.41 GB, used=65.6%, rss=176.0 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for LGBM: 0/50 evaluations
       GWO LGBM: initializing population (10 wolves)...
          fold 1/5 | acc=0.99031 | running_mean=0.99031 | time=3.65s
          fold 2/5 | acc=0.99104 | running_mean=0.99068 | time=3.55s
          fold 3/5 | acc=0.99208 | running_mean=0.99115 | time=3.58s
          fold 4/5 | acc=0.99156 | running_mean=0.99125 | time=4.88s
          fold 5/5 | acc=0.98969 | running_mean=0.99094 | time=7.69s
       GWO LGBM: init 1/10 | last=0.99094 | best=0.99094
          fold 1/5 | acc=0.98094 | running_mean=0.98094 | time=4.02s
          fold 2/5 | acc=0.98292 | running_mean=0.98193 | time=4.15s
          fold 3/5 | acc=0.98188 | running_mean=0.98191 | time=4.08s
          fold 4/5 | acc=0.98104 | running_mean=0.98169 | tim

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  n_estimators  \
 0  LGBM            Raw      0.997417              44           500   
 
    learning_rate  max_depth  num_leaves  min_child_samples  subsample  \
 0       0.355003         14          41                 67   0.882642   
 
    colsample_bytree     reg_alpha  reg_lambda  random_state  
 0          0.804129  3.282340e-07    0.002453            42  ,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0   LGBM          1  0.990938           0.990938              23.387606
 1   LGBM          2  0.981500           0.990938              20.144641
 2   LGBM          3  0.994938           0.994938              26.293972
 3   LGBM          4  0.916979           0.994938               5.263392
 4   LGBM          5  0.954313           0.994938               5.139570
 5   LGBM          6  0.980229           0.994938              17.422333
 6   LGBM          7  0.9

In [47]:
run_gwo_model("LR")


LR         | Scaler: Power      | Available RAM: 4.45 GB | Kernel RSS: 219.1 MB
       Memory cleanup -> gc_collected=223, available=4.45 GB, used=71.7%, rss=219.1 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for LR: 0/50 evaluations
       GWO LR: initializing population (10 wolves)...
          fold 1/5 | acc=0.89677 | running_mean=0.89677 | time=2.72s
          fold 2/5 | acc=0.89615 | running_mean=0.89646 | time=1.94s
          fold 3/5 | acc=0.89417 | running_mean=0.89569 | time=1.88s
          fold 4/5 | acc=0.89760 | running_mean=0.89617 | time=1.80s
          fold 5/5 | acc=0.90135 | running_mean=0.89721 | time=2.00s
       GWO LR: init 1/10 | last=0.89721 | best=0.89721
          fold 1/5 | acc=0.89687 | running_mean=0.89687 | time=1.73s
          fold 2/5 | acc=0.89615 | running_mean=0.89651 | time=1.69s
          fold 3/5 | acc=0.89354 | running_mean=0.89552 | time=1.58s
          fold 4/5 | acc=0.89750 | running_mean=0.89602 | time=1.

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration         C penalty solver  max_iter  random_state
 0    LR          Power      0.897312              44  3.339511      l1   saga      2000            42,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0     LR          1  0.897208           0.897208              10.359904
 1     LR          2  0.897062           0.897208               8.687850
 2     LR          3  0.842437           0.897208               5.141470
 3     LR          4  0.896354           0.897208               7.924488
 4     LR          5  0.854042           0.897208               5.419212
 5     LR          6  0.877021           0.897208               5.023885
 6     LR          7  0.894646           0.897208               7.002790
 7     LR          8  0.892917           0.897208               7.890523
 8     LR          9  0.895833           0.897208               9.415465
 9     LR         10

In [48]:
run_gwo_model("LDA")


LDA        | Scaler: Power      | Available RAM: 5.21 GB | Kernel RSS: 204.5 MB
       Memory cleanup -> gc_collected=223, available=5.21 GB, used=66.9%, rss=204.5 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for LDA: 0/50 evaluations
       GWO LDA: initializing population (10 wolves)...
          fold 1/5 | acc=0.89510 | running_mean=0.89510 | time=0.61s
          fold 2/5 | acc=0.89427 | running_mean=0.89469 | time=0.68s
          fold 3/5 | acc=0.89271 | running_mean=0.89403 | time=0.58s
          fold 4/5 | acc=0.89500 | running_mean=0.89427 | time=0.52s
          fold 5/5 | acc=0.89948 | running_mean=0.89531 | time=0.56s
       GWO LDA: init 1/10 | last=0.89531 | best=0.89531
          fold 1/5 | acc=0.89510 | running_mean=0.89510 | time=0.50s
          fold 2/5 | acc=0.89427 | running_mean=0.89469 | time=0.62s
          fold 3/5 | acc=0.89271 | running_mean=0.89403 | time=0.53s
          fold 4/5 | acc=0.89500 | running_mean=0.89427 | time

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration solver
 0   LDA          Power      0.895312               1  eigen,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0    LDA          1  0.895312           0.895312               2.971514
 1    LDA          2  0.895312           0.895312               2.794431
 2    LDA          3  0.895312           0.895312               5.209054
 3    LDA          4  0.895312           0.895312               5.474137
 4    LDA          5  0.895312           0.895312               5.732209
 5    LDA          6  0.895312           0.895312               5.761890
 6    LDA          7  0.895312           0.895312               6.771554
 7    LDA          8  0.895312           0.895312               6.925123
 8    LDA          9  0.895312           0.895312               6.774983
 9    LDA         10  0.895312           0.895312               5.670505
 10   LDA         11  0.895312 

In [49]:
run_gwo_model("QDA")


QDA        | Scaler: Power      | Available RAM: 7.30 GB | Kernel RSS: 198.4 MB
       Memory cleanup -> gc_collected=67, available=7.22 GB, used=54.1%, rss=198.4 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for QDA: 0/50 evaluations
       GWO QDA: initializing population (10 wolves)...
          fold 1/5 | acc=0.84292 | running_mean=0.84292 | time=1.33s
          fold 2/5 | acc=0.84406 | running_mean=0.84349 | time=1.41s
          fold 3/5 | acc=0.83948 | running_mean=0.84215 | time=1.25s
          fold 4/5 | acc=0.84062 | running_mean=0.84177 | time=1.21s
          fold 5/5 | acc=0.83719 | running_mean=0.84085 | time=1.37s
       GWO QDA: init 1/10 | last=0.84085 | best=0.84085
          fold 1/5 | acc=0.86333 | running_mean=0.86333 | time=1.31s
          fold 2/5 | acc=0.86521 | running_mean=0.86427 | time=1.27s
          fold 3/5 | acc=0.86052 | running_mean=0.86302 | time=1.12s
          fold 4/5 | acc=0.86521 | running_mean=0.86357 | time=

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  reg_param
 0   QDA          Power      0.898646              50   0.037184,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0    QDA          1  0.840854           0.840854               6.595225
 1    QDA          2  0.862438           0.862438               6.058986
 2    QDA          3  0.835333           0.862438               5.486445
 3    QDA          4  0.845375           0.862438               6.122134
 4    QDA          5  0.893625           0.893625               6.500316
 5    QDA          6  0.826354           0.893625               4.768205
 6    QDA          7  0.841583           0.893625               3.228748
 7    QDA          8  0.839896           0.893625               2.817807
 8    QDA          9  0.889542           0.893625               2.685707
 9    QDA         10  0.861542           0.893625               2.552237
 10   QDA         11  0

In [50]:
run_gwo_model("NB")


NB         | Scaler: Power      | Available RAM: 6.14 GB | Kernel RSS: 205.0 MB
       Memory cleanup -> gc_collected=64, available=6.14 GB, used=61.0%, rss=205.0 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for NB: 0/50 evaluations
       GWO NB: initializing population (10 wolves)...
          fold 1/5 | acc=0.84448 | running_mean=0.84448 | time=0.59s
          fold 2/5 | acc=0.84521 | running_mean=0.84484 | time=0.52s
          fold 3/5 | acc=0.84292 | running_mean=0.84420 | time=0.48s
          fold 4/5 | acc=0.84292 | running_mean=0.84388 | time=0.45s
          fold 5/5 | acc=0.83969 | running_mean=0.84304 | time=0.48s
       GWO NB: init 1/10 | last=0.84304 | best=0.84304
          fold 1/5 | acc=0.84448 | running_mean=0.84448 | time=0.44s
          fold 2/5 | acc=0.84521 | running_mean=0.84484 | time=0.44s
          fold 3/5 | acc=0.84292 | running_mean=0.84420 | time=0.45s
          fold 4/5 | acc=0.84292 | running_mean=0.84388 | time=0.4

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  var_smoothing
 0    NB          Power      0.843042               1   4.402874e-08,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0     NB          1  0.843042           0.843042               2.534995
 1     NB          2  0.843042           0.843042               2.203515
 2     NB          3  0.843042           0.843042               2.299188
 3     NB          4  0.843042           0.843042               2.326427
 4     NB          5  0.843042           0.843042               2.295911
 5     NB          6  0.843042           0.843042               2.280964
 6     NB          7  0.843042           0.843042               2.190794
 7     NB          8  0.843042           0.843042               2.202114
 8     NB          9  0.843042           0.843042               2.750561
 9     NB         10  0.843042           0.843042               2.295005
 10    NB      

In [56]:
apply_resource_policy_for_model('KNN')

[ResourcePolicy] model=KNN | free_ram=5.78 GB | n_jobs_cv=1 | n_jobs_opt=3


In [57]:
run_gwo_model("KNN")


KNN        | Scaler: Raw        | Available RAM: 5.78 GB | Kernel RSS: 251.7 MB
       Memory cleanup -> gc_collected=725, available=5.78 GB, used=63.2%, rss=244.1 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=3
       GWO progress for KNN: 0/50 evaluations
       GWO KNN: initializing population (10 wolves)...
          fold 1/5 | acc=0.93542 | running_mean=0.93542 | time=11.02s
          fold 2/5 | acc=0.93344 | running_mean=0.93443 | time=9.73s
          fold 3/5 | acc=0.93417 | running_mean=0.93434 | time=9.86s
          fold 4/5 | acc=0.93385 | running_mean=0.93422 | time=9.99s
          fold 5/5 | acc=0.93250 | running_mean=0.93388 | time=12.29s
       GWO KNN: init 1/10 | last=0.93388 | best=0.93388
          fold 1/5 | acc=0.92917 | running_mean=0.92917 | time=10.79s
          fold 2/5 | acc=0.93146 | running_mean=0.93031 | time=7.26s
          fold 3/5 | acc=0.93333 | running_mean=0.93132 | time=9.24s
          fold 4/5 | acc=0.93208 | running_mean=0.93151 | t

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  n_neighbors   weights     metric  p
 0   KNN            Raw      0.948208              47           16  distance  manhattan  4,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0    KNN          1  0.933875           0.933875              52.896784
 1    KNN          2  0.931729           0.933875              47.009800
 2    KNN          3  0.940333           0.940333              34.480783
 3    KNN          4  0.945979           0.945979              32.403624
 4    KNN          5  0.934875           0.945979              55.531824
 5    KNN          6  0.933646           0.945979              61.592063
 6    KNN          7  0.940083           0.945979              31.499503
 7    KNN          8  0.945875           0.945979              19.506548
 8    KNN          9  0.943500           0.945979              24.848678
 9    KNN         10  0.935604           0.9

In [51]:
run_gwo_model("LinearSVC")


LinearSVC  | Scaler: Power      | Available RAM: 6.53 GB | Kernel RSS: 209.4 MB
       Memory cleanup -> gc_collected=67, available=6.53 GB, used=58.5%, rss=209.4 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for LinearSVC: 0/50 evaluations
       GWO LinearSVC: initializing population (10 wolves)...
          fold 1/5 | acc=0.89656 | running_mean=0.89656 | time=1.16s
          fold 2/5 | acc=0.89667 | running_mean=0.89661 | time=1.07s
          fold 3/5 | acc=0.89312 | running_mean=0.89545 | time=1.04s
          fold 4/5 | acc=0.89687 | running_mean=0.89581 | time=0.95s
          fold 5/5 | acc=0.90083 | running_mean=0.89681 | time=0.93s
       GWO LinearSVC: init 1/10 | last=0.89681 | best=0.89681
          fold 1/5 | acc=0.89677 | running_mean=0.89677 | time=0.70s
          fold 2/5 | acc=0.89583 | running_mean=0.89630 | time=0.71s
          fold 3/5 | acc=0.89365 | running_mean=0.89542 | time=0.69s
          fold 4/5 | acc=0.89771 | running_me

{'status': 'completed',
 'best':        Model Optimal_Scaler  Best_CV_Mean  Best_Iteration         C   loss  dual  max_iter  random_state
 0  LinearSVC          Power      0.897125              50  5.723709  hinge  auto      5000            42,
 'history':         Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0   LinearSVC          1  0.896813           0.896813               5.177436
 1   LinearSVC          2  0.897042           0.897042               3.437735
 2   LinearSVC          3  0.865229           0.897042               2.908495
 3   LinearSVC          4  0.897000           0.897042               3.502862
 4   LinearSVC          5  0.854458           0.897042               2.679412
 5   LinearSVC          6  0.891646           0.897042               3.168312
 6   LinearSVC          7  0.896937           0.897042               3.352858
 7   LinearSVC          8  0.889250           0.897042               2.870456
 8   LinearSVC          9  0.893875       

In [77]:
run_gwo_model_with_policy("SVM")

[ResourcePolicy] model=SVM (HEAVY) | free_ram=3.07 GB | n_jobs_cv=1 | n_jobs_opt=2

SVM        | Scaler: Standard   | Available RAM: 3.07 GB | Kernel RSS: 205.5 MB
       Memory cleanup -> gc_collected=0, available=3.00 GB, used=80.9%, rss=205.6 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=2
       GWO progress for SVM: 0/50 evaluations
       GWO SVM: initializing population (10 wolves)...
          fold 1/5 | acc=0.75260 | running_mean=0.75260 | time=572.86s
          fold 2/5 | acc=0.75365 | running_mean=0.75313 | time=547.32s
          fold 3/5 | acc=0.75490 | running_mean=0.75372 | time=506.02s
          fold 4/5 | acc=0.75490 | running_mean=0.75401 | time=413.54s
          fold 5/5 | acc=0.75979 | running_mean=0.75517 | time=327.53s
       GWO SVM: init 1/10 | last=0.75517 | best=0.75517
          fold 1/5 | acc=0.73979 | running_mean=0.73979 | time=325.49s
          fold 2/5 | acc=0.73604 | running_mean=0.73792 | time=391.12s
          fold 3/5 | acc=0.74177 | r

KeyboardInterrupt: 

In [76]:
run_gwo_model_with_policy("AdaBoost")

[ResourcePolicy] model=AdaBoost (MEDIUM) | free_ram=4.37 GB | n_jobs_cv=1 | n_jobs_opt=3

AdaBoost   | Scaler: Raw        | Available RAM: 4.37 GB | Kernel RSS: 230.7 MB
       Memory cleanup -> gc_collected=1421, available=4.38 GB, used=72.2%, rss=230.7 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=3
       GWO progress for AdaBoost: 0/50 evaluations
       GWO AdaBoost: initializing population (10 wolves)...
          fold 1/5 | acc=0.87813 | running_mean=0.87813 | time=24.37s
          fold 2/5 | acc=0.87938 | running_mean=0.87875 | time=24.74s
          fold 3/5 | acc=0.87823 | running_mean=0.87858 | time=24.89s
          fold 4/5 | acc=0.87719 | running_mean=0.87823 | time=25.24s
          fold 5/5 | acc=0.87313 | running_mean=0.87721 | time=24.79s
       GWO AdaBoost: init 1/10 | last=0.87721 | best=0.87721
          fold 1/5 | acc=0.84542 | running_mean=0.84542 | time=20.42s
          fold 2/5 | acc=0.85281 | running_mean=0.84911 | time=20.49s
          fold 3/5 

{'status': 'completed',
 'best':       Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  n_estimators  learning_rate algorithm  random_state
 0  AdaBoost            Raw       0.94075               8           210         1.7124   SAMME.R            42,
 'history':        Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0   AdaBoost          1  0.877208           0.877208             124.055978
 1   AdaBoost          2  0.847250           0.877208             110.424588
 2   AdaBoost          3  0.920438           0.920438             121.257970
 3   AdaBoost          4  0.868229           0.920438              31.043243
 4   AdaBoost          5  0.923813           0.923813              99.263419
 5   AdaBoost          6  0.874844           0.923813              19.284147
 6   AdaBoost          7  0.907708           0.923813             131.942179
 7   AdaBoost          8  0.940750           0.940750              66.147841
 8   AdaBoost          9  0.863437       

In [ ]:
run_gwo_model_with_policy('RF')

[ResourcePolicy] model=RF (HEAVY) | free_ram=2.69 GB | n_jobs_cv=1 | n_jobs_opt=2

RF         | Scaler: MinMax     | Available RAM: 2.69 GB | Kernel RSS: 261.0 MB
       Memory cleanup -> gc_collected=67, available=2.67 GB, used=83.0%, rss=261.0 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=2
       GWO progress for RF: 0/50 evaluations
       GWO RF: initializing population (10 wolves)...
          fold 1/5 | acc=0.93875 | running_mean=0.93875 | time=34.69s
          fold 2/5 | acc=0.94312 | running_mean=0.94094 | time=41.21s
          fold 3/5 | acc=0.94302 | running_mean=0.94163 | time=31.37s
          fold 4/5 | acc=0.93875 | running_mean=0.94091 | time=31.79s
          fold 5/5 | acc=0.93656 | running_mean=0.94004 | time=46.25s
       GWO RF: init 1/10 | last=0.94004 | best=0.94004
          fold 1/5 | acc=0.94698 | running_mean=0.94698 | time=61.14s
          fold 2/5 | acc=0.95104 | running_mean=0.94901 | time=64.29s
          fold 3/5 | acc=0.94885 | running_mea

In [ ]:
run_gwo_model_with_policy('GB')

[ResourcePolicy] model=GB (HEAVY) | free_ram=6.09 GB | n_jobs_cv=1 | n_jobs_opt=2


In [62]:
run_gwo_model("XGB")


XGB        | Scaler: MinMax     | Available RAM: 5.22 GB | Kernel RSS: 248.6 MB
       Memory cleanup -> gc_collected=67, available=5.21 GB, used=66.9%, rss=248.6 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=2
       GWO progress for XGB: 0/50 evaluations
       GWO XGB: initializing population (10 wolves)...
          fold 1/5 | acc=0.97823 | running_mean=0.97823 | time=2.11s
          fold 2/5 | acc=0.97844 | running_mean=0.97833 | time=1.81s
          fold 3/5 | acc=0.97927 | running_mean=0.97865 | time=1.79s
          fold 4/5 | acc=0.97813 | running_mean=0.97852 | time=1.97s
          fold 5/5 | acc=0.97917 | running_mean=0.97865 | time=1.99s
       GWO XGB: init 1/10 | last=0.97865 | best=0.97865
          fold 1/5 | acc=0.97875 | running_mean=0.97875 | time=2.76s
          fold 2/5 | acc=0.98146 | running_mean=0.98010 | time=3.64s
          fold 3/5 | acc=0.98125 | running_mean=0.98049 | time=3.09s
          fold 4/5 | acc=0.98010 | running_mean=0.98039 | time=

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  n_estimators  learning_rate  max_depth  min_child_weight  subsample  colsample_bytree     gamma     reg_alpha    reg_lambda  random_state
 0   XGB         MinMax      0.993646              34           427            0.5          4                 1   0.756243          0.920541  0.331947  1.374810e-07  6.711895e-07            42,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0    XGB          1  0.978646           0.978646               9.686611
 1    XGB          2  0.980208           0.980208              15.212054
 2    XGB          3  0.979625           0.980208               8.686960
 3    XGB          4  0.933375           0.980208               6.142359
 4    XGB          5  0.955188           0.980208               3.052778
 5    XGB          6  0.976604           0.980208               9.433269
 6    XGB          7  0.979479           0.980208          

In [46]:
run_gwo_model("CB")


CB         | Scaler: MinMax     | Available RAM: 2.48 GB | Kernel RSS: 125.5 MB
       Memory cleanup -> gc_collected=0, available=2.42 GB, used=84.6%, rss=192.2 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=8
       GWO progress for CB: 0/50 evaluations
       GWO CB: initializing population (10 wolves)...
          fold 1/5 | acc=0.98333 | running_mean=0.98333 | time=11.40s
          fold 2/5 | acc=0.98219 | running_mean=0.98276 | time=12.53s
          fold 3/5 | acc=0.98823 | running_mean=0.98458 | time=12.27s
          fold 4/5 | acc=0.98740 | running_mean=0.98529 | time=12.71s
          fold 5/5 | acc=0.98354 | running_mean=0.98494 | time=12.64s
       GWO CB: init 1/10 | last=0.98494 | best=0.98494
          fold 1/5 | acc=0.98250 | running_mean=0.98250 | time=5.97s
          fold 2/5 | acc=0.98271 | running_mean=0.98260 | time=5.92s
          fold 3/5 | acc=0.98510 | running_mean=0.98344 | time=5.27s
          fold 4/5 | acc=0.98479 | running_mean=0.98378 | time

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration  iterations  \
 0    CB         MinMax      0.994438              35         361   
 
    learning_rate  depth  l2_leaf_reg  border_count  bagging_temperature  \
 0            0.5      6         10.0           251             2.192955   
 
    random_state  
 0            42  ,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0     CB          1  0.984938           0.984938              61.591209
 1     CB          2  0.983563           0.984938              27.301666
 2     CB          3  0.989375           0.989375              28.645044
 3     CB          4  0.989583           0.989583              59.832264
 4     CB          5  0.948292           0.989583              46.088523
 5     CB          6  0.990500           0.990500              44.062657
 6     CB          7  0.917563           0.990500               6.660882
 7     CB          8  0.976187          

In [60]:
run_gwo_model("SGD")


SGD        | Scaler: Power      | Available RAM: 6.50 GB | Kernel RSS: 227.3 MB
       Memory cleanup -> gc_collected=67, available=6.50 GB, used=58.7%, rss=227.3 MB
       Running 50 evaluations with n_jobs_cv=1, n_jobs_opt=3
       GWO progress for SGD: 0/50 evaluations
       GWO SGD: initializing population (10 wolves)...
          fold 1/5 | acc=0.87406 | running_mean=0.87406 | time=0.54s
          fold 2/5 | acc=0.88021 | running_mean=0.87714 | time=0.50s
          fold 3/5 | acc=0.87438 | running_mean=0.87622 | time=0.55s
          fold 4/5 | acc=0.87760 | running_mean=0.87656 | time=0.48s
          fold 5/5 | acc=0.87719 | running_mean=0.87669 | time=0.52s
       GWO SGD: init 1/10 | last=0.87669 | best=0.87669
          fold 1/5 | acc=0.86125 | running_mean=0.86125 | time=0.52s
          fold 2/5 | acc=0.86760 | running_mean=0.86443 | time=0.54s
          fold 3/5 | acc=0.86146 | running_mean=0.86344 | time=0.54s
          fold 4/5 | acc=0.86438 | running_mean=0.86367 | time=

{'status': 'completed',
 'best':   Model Optimal_Scaler  Best_CV_Mean  Best_Iteration            loss penalty    alpha  l1_ratio  max_iter  random_state
 0   SGD          Power      0.895542              28  modified_huber      l1  0.00091  0.630203      2000            42,
 'history':    Model  Iteration   CV_Mean  Best_Score_So_Far  Iteration_Runtime_Sec
 0    SGD          1  0.876688           0.876688               2.609132
 1    SGD          2  0.863312           0.876688               2.659617
 2    SGD          3  0.893396           0.893396               2.852707
 3    SGD          4  0.891646           0.893396               3.294501
 4    SGD          5  0.871333           0.893396               2.515096
 5    SGD          6  0.865021           0.893396               2.503521
 6    SGD          7  0.891833           0.893396               3.253172
 7    SGD          8  0.866396           0.893396               2.699696
 8    SGD          9  0.887687           0.893396        

In [ ]:
# =====================================================================
# CELL 9: Final Aggregation with Comprehensive Analysis & Resource Efficiency Report
# =====================================================================

print("\n" + "="*80)
print("FINAL AGGREGATION & RESOURCE EFFICIENCY ANALYSIS")
print("="*80)

# =====================================================================
# 0. Resource Efficiency Report
# =====================================================================

final_memory = psutil.Process(os.getpid()).memory_info().rss / (1024**2)
peak_memory_percent = (final_memory / (TOTAL_MEMORY_GB * 1024)) * 100

print(f"\nSystem resource usage:")
print(f"  Final Memory:            {final_memory:.1f} MB ({peak_memory_percent:.1f}% of total)")
print(f"  CPU Cores Utilized:      {N_JOBS_CV} (from {CPU_COUNT} available)")
print(f"  Adaptive Mode:           {'ON' if AVAILABLE_MEMORY_GB < 12 else 'Optimized'}")
print(f"  Garbage Collection:      Enabled (post-model cleanup)")

# =====================================================================
# 1. Reload timing data if cells were run separately
# =====================================================================

if 'tpe_timing_df' not in dir():
    tpe_timing_file = RESULTS_DIR / "tpe_timing_per_model.csv"
    if tpe_timing_file.exists():
        tpe_timing_df = pd.read_csv(tpe_timing_file)
        print("\n✓ Loaded TPE timing from file")
else:
    print("\n✓ TPE timing in memory")

if 'gwo_timing_df' not in dir():
    gwo_timing_file = RESULTS_DIR / "gwo_timing_per_model.csv"
    if gwo_timing_file.exists():
        gwo_timing_df = pd.read_csv(gwo_timing_file)
        print("✓ Loaded GWO timing from file")
else:
    print("✓ GWO timing in memory")

# =====================================================================
# 2. Create comprehensive optimization comparison with efficiency metrics
# =====================================================================

if 'tpe_timing_df' in dir() and 'gwo_timing_df' in dir():
    comparison_timing = pd.merge(
        tpe_timing_df[["Model", "TPE_Search_Sec", "TPE_Evaluation_Sec", "TPE_Total_Sec", "Best_CV_Mean"]].rename(columns={"Best_CV_Mean": "TPE_Best_CV"}),
        gwo_timing_df[["Model", "GWO_Search_Sec", "GWO_Evaluation_Sec", "GWO_Total_Sec", "Best_CV_Mean"]].rename(columns={"Best_CV_Mean": "GWO_Best_CV"}),
        on="Model",
        how="outer"
    )
    
    # Efficiency metrics
    comparison_timing["TPE_Faster"] = comparison_timing["TPE_Total_Sec"] < comparison_timing["GWO_Total_Sec"]
    comparison_timing["Time_Difference_Sec"] = (comparison_timing["GWO_Total_Sec"] - comparison_timing["TPE_Total_Sec"]).abs()
    comparison_timing["CV_Improvement"] = (comparison_timing["GWO_Best_CV"] - comparison_timing["TPE_Best_CV"]) * 100
    # Efficiency: improvement per second
    comparison_timing["Efficiency_Score"] = comparison_timing["CV_Improvement"] / (comparison_timing["Time_Difference_Sec"] + 1e-6)
    
    comparison_timing.to_csv(RESULTS_DIR / "optimization_method_comparison_timing.csv", index=False)
    
    print("\n" + "="*80)
    print("OPTIMIZATION METHOD COMPARISON (Timing, Performance & Efficiency)")
    print("="*80)
    print(comparison_timing[["Model", "TPE_Total_Sec", "GWO_Total_Sec", "TPE_Faster", "TPE_Best_CV", "GWO_Best_CV", "CV_Improvement", "Efficiency_Score"]].to_string(index=False))
    
    # Summary statistics with efficiency analysis
    print("\n" + "-"*80)
    print("SUMMARY STATISTICS & EFFICIENCY METRICS:")
    print("-"*80)
    total_tpe_time = comparison_timing['TPE_Total_Sec'].sum()
    total_gwo_time = comparison_timing['GWO_Total_Sec'].sum()
    total_cv_improvement = comparison_timing['CV_Improvement'].mean()
    
    print(f"Total TPE time:          {total_tpe_time:8.2f}s ({total_tpe_time/60:6.2f} min)")
    print(f"Total GWO time:          {total_gwo_time:8.2f}s ({total_gwo_time/60:6.2f} min)")
    print(f"Time ratio (GWO/TPE):    {total_gwo_time/total_tpe_time:8.2f}x")
    print(f"TPE mean time/model:     {comparison_timing['TPE_Total_Sec'].mean():8.2f}s")
    print(f"GWO mean time/model:     {comparison_timing['GWO_Total_Sec'].mean():8.2f}s")
    print(f"Mean CV improvement:     {total_cv_improvement:+.4f} pp (GWO vs TPE)")
    print(f"Models faster with TPE:  {(~comparison_timing['TPE_Faster']).sum():2d} / {len(comparison_timing)}")
    print(f"Efficiency winner:       {'TPE' if total_tpe_time < total_gwo_time else 'GWO'}")

# =====================================================================
# 3. Inference Time Analysis (critical for deployment)
# =====================================================================

print("\n" + "="*80)
print("INFERENCE TIME ANALYSIS (Per Sample)")
print("="*80)

inference_timing_records = []

if not tpe_results_df.empty:
    tpe_test_df = tpe_results_df[tpe_results_df["Split"] == "Test"].copy()
    for _, row in tpe_test_df.iterrows():
        model_code = row["Model"]
        params = {k: v for k, v in row.items() if k not in ["Method", "Model", "Optimal_Scaler", "Split", "Accuracy", "Precision", "Recall", "F1", "AUC", "LogLoss", "Kappa", "MCC"] + [c for c in row.index if "CI_" in c]}
        
        pipeline = make_pipeline(model_code, params, row["Optimal_Scaler"])
        pipeline.fit(X_train, y_train)
        
        # Time inference on test set (3 runs for stability)
        times = []
        for _ in range(3):
            inference_start = time.perf_counter()
            _ = pipeline.predict(X_test)
            times.append(time.perf_counter() - inference_start)
        
        inference_time_total = np.mean(times)
        inference_time_per_sample_us = (inference_time_total / len(X_test)) * 1_000_000
        
        inference_timing_records.append({
            "Method": "TPE",
            "Model": model_code,
            "Optimal_Scaler": row["Optimal_Scaler"],
            "Inference_Time_Total_Sec": float(inference_time_total),
            "Inference_Time_Per_Sample_Us": float(inference_time_per_sample_us),
            "Test_Samples": len(X_test),
            "CV_Accuracy": row["Accuracy"],
        })

if not gwo_results_df.empty:
    gwo_test_df = gwo_results_df[gwo_results_df["Split"] == "Test"].copy()
    for _, row in gwo_test_df.iterrows():
        model_code = row["Model"]
        params = {k: v for k, v in row.items() if k not in ["Method", "Model", "Optimal_Scaler", "Split", "Accuracy", "Precision", "Recall", "F1", "AUC", "LogLoss", "Kappa", "MCC"] + [c for c in row.index if "CI_" in c]}
        
        pipeline = make_pipeline(model_code, params, row["Optimal_Scaler"])
        pipeline.fit(X_train, y_train)
        
        # Time inference (3 runs for stability)
        times = []
        for _ in range(3):
            inference_start = time.perf_counter()
            _ = pipeline.predict(X_test)
            times.append(time.perf_counter() - inference_start)
        
        inference_time_total = np.mean(times)
        inference_time_per_sample_us = (inference_time_total / len(X_test)) * 1_000_000
        
        inference_timing_records.append({
            "Method": "GWO",
            "Model": model_code,
            "Optimal_Scaler": row["Optimal_Scaler"],
            "Inference_Time_Total_Sec": float(inference_time_total),
            "Inference_Time_Per_Sample_Us": float(inference_time_per_sample_us),
            "Test_Samples": len(X_test),
            "CV_Accuracy": row["Accuracy"],
        })

inference_timing_df = pd.DataFrame(inference_timing_records)
inference_timing_df.to_csv(RESULTS_DIR / "inference_timing_analysis.csv", index=False)

if not inference_timing_df.empty:
    print("\nInference Timing Summary:")
    print(inference_timing_df[["Method", "Model", "Inference_Time_Per_Sample_Us", "Inference_Time_Total_Sec", "CV_Accuracy"]].to_string(index=False))
    
    print("\n" + "-"*80)
    print("INFERENCE PERFORMANCE METRICS:")
    print("-"*80)
    for method in ["TPE", "GWO"]:
        subset = inference_timing_df[inference_timing_df["Method"] == method]
        if not subset.empty:
            print(f"{method} Method:")
            print(f"  Mean inference time per sample: {subset['Inference_Time_Per_Sample_Us'].mean():8.2f} µs")
            print(f"  Min  inference time per sample: {subset['Inference_Time_Per_Sample_Us'].min():8.2f} µs ({subset.loc[subset['Inference_Time_Per_Sample_Us'].idxmin(), 'Model']})")
            print(f"  Max  inference time per sample: {subset['Inference_Time_Per_Sample_Us'].max():8.2f} µs ({subset.loc[subset['Inference_Time_Per_Sample_Us'].idxmax(), 'Model']})")
            print()

# =====================================================================
# 4. Consolidate all results (Baseline, TPE, GWO)
# =====================================================================

all_results_frames = [baseline_results_df]
if not gwo_results_df.empty:
    all_results_frames.append(gwo_results_df)
if not tpe_results_df.empty:
    all_results_frames.append(tpe_results_df)

all_results_df = pd.concat(all_results_frames, ignore_index=True)
all_results_df["Method"] = pd.Categorical(all_results_df["Method"], categories=METHOD_ORDER, ordered=True)
all_results_df["Model"] = pd.Categorical(all_results_df["Model"], categories=MODEL_ORDER, ordered=True)
all_results_df = all_results_df.sort_values(["Split", "Model", "Method"]).reset_index(drop=True)

comparison_table = all_results_df[["Method", "Model", "Optimal_Scaler", "Split", *METRICS]].copy()
comparison_table.to_csv(RESULTS_DIR / "all_methods_results_point_estimates.csv", index=False)
all_results_df.to_csv(RESULTS_DIR / "all_methods_results_with_ci.csv", index=False)

ci_long_rows = []
for _, row in all_results_df.iterrows():
    for metric_name in METRICS:
        ci_long_rows.append(
            {
                "Method": row["Method"],
                "Model": row["Model"],
                "Optimal_Scaler": row["Optimal_Scaler"],
                "Split": row["Split"],
                "Metric": metric_name,
                "Point_Estimate": row[metric_name],
                "CI_Lower": row[f"{metric_name}_CI_Lower"],
                "CI_Upper": row[f"{metric_name}_CI_Upper"],
            }
        )

ci_long_df = pd.DataFrame(ci_long_rows)
ci_long_df.to_csv(RESULTS_DIR / "all_methods_metric_ci_long.csv", index=False)

run_metadata = {
    "data_path": str(DATA_PATH),
    "results_dir": str(RESULTS_DIR),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "n_splits": N_SPLITS,
    "n_trials_per_model": N_TRIALS_PER_MODEL,
    "gwo_population_size": GWO_POPULATION_SIZE,
    "gwo_iterations": GWO_ITERATIONS,
    "n_bootstrap": N_BOOTSTRAP,
    "selected_feature_count": len(selected_features),
    "selected_features": selected_features,
    "optimized_models": sorted(list(SEARCH_SPACES.keys())),
    "all_models": MODEL_ORDER,
    "system_resources": {
        "cpu_cores": CPU_COUNT,
        "total_memory_gb": TOTAL_MEMORY_GB,
        "available_memory_gb": AVAILABLE_MEMORY_GB,
        "n_jobs_cv": N_JOBS_CV,
        "n_jobs_optimization": N_JOBS_OPT,
    },
    "optimization_note": "Separate TPE and GWO cells with adaptive parallelization and resource monitoring",
}
with open(RESULTS_DIR / "run_metadata.json", "w", encoding="utf-8") as handle:
    json.dump(run_metadata, handle, indent=2)

gc.collect()  # Final cleanup

print("\n" + "="*80)
print("✓ UNIFIED PIPELINE COMPLETED SUCCESSFULLY")
print("="*80)
print(f"Point estimates:          {RESULTS_DIR / 'all_methods_results_point_estimates.csv'}")
print(f"Results with CI:          {RESULTS_DIR / 'all_methods_results_with_ci.csv'}")
print(f"Long-format CI:           {RESULTS_DIR / 'all_methods_metric_ci_long.csv'}")
print(f"Timing comparison:        {RESULTS_DIR / 'optimization_method_comparison_timing.csv'}")
print(f"Inference analysis:       {RESULTS_DIR / 'inference_timing_analysis.csv'}")
print(f"TPE timing per model:     {RESULTS_DIR / 'tpe_timing_per_model.csv'}")
print(f"GWO timing per model:     {RESULTS_DIR / 'gwo_timing_per_model.csv'}")
print("="*80)